# 3D bSSFP (alpha=60) IR Kooshball - kwic view-sharing + Ehses T1/T2/PD


In [ ]:
%matplotlib widget

import os
import sys
import time
from typing import Optional, Tuple

import matplotlib.pyplot as plt
import MRzeroCore as mr0
import itertools
import numpy as np
import pypulseq as pp
from pypulseq import Sequence
import torch
import torch.nn as nn
import torchkbnufft as tkbn
from Modules.CoilSens import cyclic_head_coil
from Modules.FourierTransform import (
    nonuniform_fourier_transform_adjoint,
    nonuniform_fourier_transform_forward,
)
from Modules.Linops import ChannelOperator, LinearOperator
from Modules.NumSolvers import conjugate_gradient
from Modules.fast_iso import isochromat_sim_fast
from Modules.Plan import KbNufft, KbNufftAdjoint
from tqdm import tqdm

sys.path.insert(0, "..")
from Modules import default_system, load_brain_phantom

torch.set_default_device("cuda")
device = torch.device("cuda")


## 1. Sequence builder (bSSFP, constant 180-deg RF cycling)


In [ ]:
def build_bssfp(
    FOV: float = 250e-3,
    N_x: int = 64,
    alpha: float = 60.0,
    N_channels = 8,
    slice_thickness: float = 64e-3,
    TE: float = 3.98e-3,
    N_readout: int = 384,
    N_segments: int = 25,
    N_dummy: int = 0,
    TI_start: float = 20e-3,
    TI_inter: float = 4.88e-3,
    TD: float = 2000e-3,
    rf_spoiling_inc: float = 180.0,
    write_seq: bool = False,
    plot: bool = False,
) -> pp.Sequence:
    """IR balanced-SSFP (bSSFP, alpha = 60 deg) 3D radial "Kooshball" sequence.

    25 inversion segments, each: inversion -> TI_start -> N_readout bSSFP
    echoes -> TD, with a golden-angle radial readout per echo.

    RF cycling (the key physics): the excitation/ADC phase offset advances by a
    CONSTANT rf_spoiling_inc (default 180 deg) per echo.  That is the
    steady-state phase cycle of a bSSFP acquisition.  With 180 deg and an even
    N_readout, echo p of EVERY segment has the same RF phase, so the
    per-echo-index frames used for the T1* fit are mutually coherent.

    (The MPnRAGE builder instead used a *growing* increment - the rf phase
    accumulated as the sum of a linearly increasing increment - which spoiled
    the transverse steady state and destroyed the per-echo coherence.)
    """
    system = default_system()
    seq = pp.Sequence(system)
    deltak = 1 / FOV

    rf_inv, gz_inv, _ = pp.make_sinc_pulse(
        flip_angle=np.pi, duration=10e-3, slice_thickness=slice_thickness,
        apodization=0.5, time_bw_product=4, system=system,
        return_gz=True, use='inversion')

    rf, gz, _ = pp.make_sinc_pulse(
        apodization=0.5, duration=2e-3, flip_angle=alpha*np.pi/180,
        slice_thickness=slice_thickness, system=system,
        time_bw_product=4, return_gz=True, use='excitation')

    gx_base = pp.make_trapezoid(channel='x', flat_area=N_x*deltak, flat_time=6.4e-3/5, system=system)
    adc = pp.make_adc(num_samples=N_x, duration=gx_base.flat_time, delay=gx_base.rise_time, system=system)
    gx_pre_base = pp.make_trapezoid(channel='x', area=-gx_base.area/2, duration=2e-3, system=system)
    gz_reph = pp.make_trapezoid(channel='z', area=-gz.area/2, duration=2e-3, system=system)

    # z gradient played right after the inversion pulse. It cancels the
    # slab-select area of the inversion (gz_inv.area) and leaves -gz.area/2,
    # i.e. exactly the pre-compensation the first excitation needs.
    gz_post_inv = pp.make_trapezoid(
        channel='z', area=-gz_inv.area - gz.area/2, system=system)

    delay_TE = (
        np.ceil((
            TE - pp.calc_duration(gx_pre_base)
            - gz.fall_time - gz.flat_time/2
            - pp.calc_duration(gx_base)/2
        ) / seq.grad_raster_time) * seq.grad_raster_time
    )
    inv_duration   = pp.calc_duration(rf_inv, gz_inv)
    delay_TI_start = (
        np.ceil((TI_start - inv_duration/2 - pp.calc_duration(gz)/2)
                / seq.grad_raster_time) * seq.grad_raster_time
    )
    assert delay_TI_start > 0, 'TI_start too short'
    assert pp.calc_duration(gz_post_inv) <= delay_TI_start, 'post-inversion z gradient does not fit in TI_start'
    delay_TI_inter = np.ceil(TI_inter / seq.grad_raster_time) * seq.grad_raster_time

    gre_block_dur = (
        pp.calc_duration(gz) + pp.calc_duration(gx_pre_base)
        + delay_TE + pp.calc_duration(gx_base)
        + pp.calc_duration(gx_pre_base)
    )
    delay_TD = np.ceil(TD / seq.grad_raster_time) * seq.grad_raster_time

    total_acquired = N_segments * N_readout
    def spoke_dir(idx):
        phi = np.arccos(
            1 - 2*(idx + 0.5)/total_acquired)
        theta = np.pi*(1 + np.sqrt(5))*idx
    
        return (np.sin(phi)*np.cos(theta), np.sin(phi)*np.sin(theta), np.cos(phi))

    rf_phase = 0.0
    spoke_counter = 0

    for seg in range(N_segments):
        seq.add_block(rf_inv, gz_inv)
        seq.add_block(pp.make_delay(delay_TI_start), gz_post_inv)   # z balance after inversion

        n_this_seg = N_readout + (N_dummy if seg == 0 else 0)
        for s in range(n_this_seg):
            is_dummy = (seg == 0) and (s < N_dummy)
            is_last  = (s == n_this_seg - 1)

            rf.phase_offset  = rf_phase / 180 * np.pi
            adc.phase_offset = rf_phase / 180 * np.pi
            # CONSTANT bSSFP phase cycle (default 180 deg/echo) - NOT the
            # MPnRAGE growing increment.
            rf_phase = (rf_phase + rf_spoiling_inc) % 360

            kx, ky, kz = (1.0, 0.0, 0.0) if is_dummy else spoke_dir(spoke_counter)
            if not is_dummy:
                spoke_counter += 1

            seq.add_block(rf, gz)
            seq.add_block(
                pp.make_trapezoid(channel='x', area=gx_pre_base.area*kx, duration=2e-3, system=system),
                pp.make_trapezoid(channel='y', area=gx_pre_base.area*ky, duration=2e-3, system=system),
                pp.make_trapezoid(channel='z', area=gx_pre_base.area*kz+gz_reph.area, duration=2e-3, system=system),
            )
            seq.add_block(pp.make_delay(delay_TE))

            gx = pp.make_trapezoid(channel='x', amplitude=gx_base.amplitude*kx,
                        flat_time=gx_base.flat_time, rise_time=gx_base.rise_time, fall_time=gx_base.fall_time, system=system)
            gy = pp.make_trapezoid(channel='y', amplitude=gx_base.amplitude*ky,
                        flat_time=gx_base.flat_time, rise_time=gx_base.rise_time, fall_time=gx_base.fall_time, system=system)
            gz_ro = pp.make_trapezoid(channel='z', amplitude=gx_base.amplitude*kz,
                        flat_time=gx_base.flat_time, rise_time=gx_base.rise_time, fall_time=gx_base.fall_time, system=system)

            if is_dummy: seq.add_block(gx, gy, gz_ro)
            else: seq.add_block(gx, gy, gz_ro, adc)

            # rewinder: x/y undo the prephaser; z undoes the kz blip and
            # pre-compensates the next slab-select lobe (-gz.area/2), except in
            # the last TR of a segment, where the z moment must end at 0.
            seq.add_block(
                pp.make_trapezoid(channel='x', area=gx_pre_base.area*kx, duration=2e-3, system=system),
                pp.make_trapezoid(channel='y', area=gx_pre_base.area*ky, duration=2e-3, system=system),
                pp.make_trapezoid(channel='z', area=gx_pre_base.area*kz + (0.0 if is_last else gz_reph.area),
                                  duration=2e-3, system=system),
            )
            if s < n_this_seg - 1:
                seq.add_block(pp.make_delay(delay_TI_inter))

        seq.add_block(pp.make_delay(delay_TD))

    ok, err = seq.check_timing()
    if ok:
        print('bSSFP timing check passed')
        print(f'  GRE block dur : {gre_block_dur*1000:.1f} ms')
        print(f'  Echo spacing P: {(gre_block_dur+delay_TI_inter)*1000:.1f} ms')
        print(f'  Readout train : {N_readout*(gre_block_dur+delay_TI_inter)*1000:.0f} ms')
        est = N_segments*(inv_duration+delay_TI_start+N_readout*gre_block_dur
                          +(N_readout-1)*delay_TI_inter+TD)
        print(f'  Scan time est : {est/60:.1f} min')
    else: print(f'Timing FAILED: {err}')

    seq.set_definition('FOV',  [FOV, FOV, slice_thickness])
    seq.set_definition("Name", "bssfp_kooshball3D_signal_n64_R384_S25.seq")
    if write_seq:
        seq.write("bssfp_kooshball3D_signal_n64_R384_S25.seq")
    if plot:
        seq.plot(time_range=(0.2, 0.4))
    return seq, gre_block_dur, delay_TI_inter, TD

## 2. Phantom, tissues, masks, coils


In [ ]:
device = torch.device("cuda")

# ---- single source of truth ------------------------------------------------
FOV        = 250e-3
n          = 64      # phantom + recon matrix (n^3)
N_x        = n       # readout samples / spoke  -> kmax matches the recon grid
N_readout  = 384     # echoes per inversion segment
N_segments = 25      # inversion segments (25*384 = 9600 spokes)
n_adc      = N_segments * N_readout * N_x   # total ADC samples (8 coils apart)
N_dummy    = 0
slice_thickness = 64e-3
TI_START   = 20e-3    # TI of the first echo; the train then spans ~20 ms -> ~3.1 s
TI_inter   = 4.88e-3  # recovery delay between echoes (echo spacing P = TR + TI_inter)
N_channels = 8
spin_count = 64       # isochromats per voxel (simulation only)

# Tissue ground truth (as defined in Modules/phantom.py):
#   label 1 -> CSF: PD 1.0, T1 4.0 s, T2 0.5 s
#   label 2 -> GM : PD 0.8, T1 1.3 s, T2 0.08 s
#   label 3 -> WM : PD 0.7, T1 0.9 s, T2 0.07 s
#   T2* = 0.04 s everywhere
TISSUES = {
    "WM":  dict(T1=0.9, T2=0.07, PD=0.7),
    "GM":  dict(T1=1.3, T2=0.08, PD=0.8),
    "CSF": dict(T1=4.0, T2=0.5,  PD=1.0),
}
T2STAR = 0.04
NAMES = ("WM", "GM", "CSF")

# ---- phantom (BrainWeb subject04) ------------------------------------------
phantom = load_brain_phantom(
    raw_path="subject04_crisp_v.rawb",
    target_size=(n, n, n),
    device=device,
    permute_order=(2, 1, 0),
)
print(type(phantom).__name__, "size =", phantom.size)

# Tissue masks on the recon grid, from the phantom PD map (identity orientation
# - the recon grid matches the phantom directly, no flip).  Thresholds chosen
# so the interpolated PD separates the three labels.
pd_map = phantom.PD.detach().cpu().numpy()
# 72% of the volume is background with pd ~ 0; it MUST be excluded from
# every mask.  The old WM mask (pd < 0.75) accidentally included all of the
# background (187,967 air voxels), which is why the "WM" transient looked
# flat - it was ~87% empty air.  GM/CSF were already background-free.
brain = pd_map > 0.10                       # exclude background + thin rim
masks = {
    "WM":  brain & (pd_map < 0.75),
    "GM":  (pd_map >= 0.75) & (pd_map < 0.90),
    "CSF": (pd_map >= 0.90),
}
brain_mask = brain
for t in NAMES:
    print(f"mask {t}: {int(masks[t].sum())} voxels")

# ---- coils (used directly - identity orientation, no auto-match) -----------
scale = tuple(float(s) / FOV for s in phantom.size)
coil_sim = cyclic_head_coil((N_channels, n, n, n), fov_scaling=scale).to(torch.complex64)
phantom.coil_sens = coil_sim.to(device)
phantom.B0 = 0 * phantom.B0
phantom.B1 = torch.ones(phantom.B1.shape, dtype=phantom.B1.dtype)
data = phantom.build()
print("simulation voxels:", int(data.PD.numel()))

# combine weights for the SENSE/adaptive combine of the NUFFT previews
coil_c64 = coil_sim.to(device)
coil_den = (coil_c64.abs() ** 2).sum(0)


## 3. Build sequence + timing + null TIs


In [ ]:
# ---- build the bSSFP sequence (constant 180 deg RF cycling) -----------------
t1 = time.time()
seq, TR_echo, delay_TI_inter, TD = build_bssfp(
    FOV=FOV, N_x=N_x, N_channels=N_channels, slice_thickness=slice_thickness,
    N_readout=N_readout, N_segments=N_segments, N_dummy=N_dummy,
    TI_start=TI_START, TI_inter=TI_inter, write_seq=True, plot=True,
    rf_spoiling_inc=180.0,          # bSSFP steady-state cycle (was 117.0 growing)
)
P_echo = TR_echo + delay_TI_inter
alpha = 60.0 * np.pi / 180
print(f"build: {time.time()-t1:.0f} s   P = {P_echo*1e3:.2f} ms")

# effective TI of echo index p (p = 0..N_readout-1): the train starts at
# TI_START and each echo index advances by one echo spacing P.
ti_effs = np.array([TI_START + p * P_echo for p in range(N_readout)])

def null_ti(T1, P, TR=TR_echo, TD=TD, alpha=alpha, N=N_readout):
    # TI at which the first excitation of each (cyclic) segment sees Mz = 0.
    # Cyclic state: the magnetization just before the inversion pulse is the
    # fixed point of one segment (inversion -> TI -> N echoes -> TD).
    E1 = np.exp(-P / T1)
    lam = E1 * np.cos(alpha)
    kap = (1 - E1) / (1 - E1 * np.cos(alpha))          # Mz_ss / M0
    E_tot = np.exp(-(TR + TD) / T1)
    z_star = (1 - E_tot) + E_tot * np.cos(alpha) * kap * (1 - lam ** (N - 1))
    return T1 * np.log(1.0 + z_star)

TI_null = {t: null_ti(p["T1"], P_echo) for t, p in TISSUES.items()}
null_frame = {t: int(np.argmin(np.abs(ti_effs - TI_null[t]))) for t in TI_null}
for t in NAMES:
    print(f"TI_null({t}) = {TI_null[t]*1e3:7.1f} ms   frame {null_frame[t]:3d} "
          f"(TI_eff = {ti_effs[null_frame[t]]*1e3:5.0f} ms)")

## 4. Simulated signal (load, or re-simulate)


In [ ]:
t1 = time.time()
mr_seq = mr0.Sequence.import_file("bssfp_kooshball3D_signal_n64_R384_S25.seq")

In [ ]:
# ---- simulated signal -------------------------------------------------------
# A 180-deg-cycling simulation of this exact sequence has already been run;
# load it (fast).  Set RESIMULATE = True to re-run from scratch (~5 min on the
# A100).  The old MPnRAGE_kooshball3D_signal_*.pt / .seq files (117-deg growing
# spoiling) are IR-MPRAGE-style spoiled data - NOT valid for this bSSFP
# quantification - and are deliberately not used.

signal = isochromat_sim_fast(mr_seq, data.cuda(), spin_count=spin_count, spin_dist="rand", print_progress=True, chunk_bytes=2**30)
signal = signal.to(device).permute((1, 0)).contiguous().to(torch.complex64)

assert tuple(signal.shape) == (N_channels, n_adc), (tuple(signal.shape), (N_channels, n_adc))
print("simulated signal:", tuple(signal.shape))

## 5. Trajectory, NUFFT preview, per-echo-index transient


In [ ]:
# ---- trajectory + fast NUFFT preview recon ---------------------------------
def kspace_to_rad(mr_seq, n, FOV):
    'ADC k-space (1/m) -> rad/pixel of an n^3 grid spanning FOV.  Returns (3, N_adc).'
    k = torch.as_tensor(mr_seq.get_kspace()).float()
    k = k.reshape(-1, k.shape[-1])[:, :3].T.contiguous()
    return k * (2 * np.pi * FOV / n)

k_full = kspace_to_rad(mr_seq, n, FOV)
assert k_full.shape[1] == signal.shape[1], (k_full.shape, signal.shape)
k_full = k_full.to(device)

# samples with |k| > pi lie beyond the grid's kmax and must be DROPPED (not
# clamped) for the gridding recon; CG uses the full set with k normalized to
# [-0.5, 0.5] cycles/voxel.
keep_full = k_full.norm(dim=0) <= np.pi
k_norm = k_full / k_full.norm(dim=0).max() * 0.5     # cycles/voxel, for the CG operator
print(f"keep {int(keep_full.sum())}/{keep_full.numel()} samples (|k| <= pi)")

def nufft_recon(sig, k_rad, n):
    'Per-channel density-compensated adjoint NUFFT.  sig: (C, N), k_rad: (3, N) -> (C, n, n, n).'
    keep = k_rad.norm(dim=0) <= np.pi
    k, s = k_rad[:, keep], sig[:, keep]
    dcomp = k.norm(dim=0).clamp(min=np.pi / n) ** 2   # |k|^2 for radial
    adj = tkbn.KbNufftAdjoint(im_size=(n, n, n), grid_size=(2 * n,) * 3).to(sig.device)
    return adj((s * dcomp)[None], k)[0]

def combine(img_c):
    'adaptive SENSE combine of a (C, n, n, n) multi-coil image -> (n, n, n) complex'
    return (coil_c64.conj() * img_c).sum(0) / coil_den.clamp(min=1e-2 * coil_den.max())

# ---- per-echo-index frames (normal NUFFT previews) --------------------------
# Echo p of every segment fired TI_eff(p) after that segment's inversion, so
# each frame is the brain at a single effective TI.  Frame p = the p-th echo of
# all N_segments (25 spokes / frame).  A quick look at the transient:
def frame_spoke_blocks(p):
    'ADC block (N_x samples) index of echo p in every segment'
    return [s * N_readout + p for s in range(N_segments)]

def frame_idx(p):
    return torch.cat([torch.arange(b * N_x, (b + 1) * N_x) for b in frame_spoke_blocks(p)])

t1 = time.time()
preview = {}
for p in range(0, N_readout, 24):        # 16 frames
    idx = frame_idx(p)
    preview[p] = combine(nufft_recon(signal[:, idx], k_full[:, idx], n))
print(f"preview recon ({len(preview)} frames): {time.time()-t1:.0f} s")

# signed (steady-state phase-referenced) tissue-mean transient
ref_phase = torch.angle(preview[max(preview)])
tvals = np.array([ti_effs[p] for p in sorted(preview)])
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
mid = n // 2
ax[0].imshow(np.rot90(preview[168][:, :, mid].T.abs().cpu().numpy(), 2), cmap="gray")
ax[0].set_title(f"NUFFT preview, frame 168\nTI = {ti_effs[168]*1e3:.0f} ms")
ax[0].axis("off")
for t in NAMES:
    m = torch.as_tensor(masks[t], device=device)
    s = [float(torch.real(preview[p] * torch.exp(-1j * ref_phase))[m].mean())
         for p in sorted(preview)]
    ax[1].plot(tvals * 1e3, s, label=t)
ax[1].axhline(0, color="k", lw=0.5)
ax[1].set_title("signed tissue-mean transient (SS-phase referenced)")
ax[1].set_xlabel("TI (ms)"); ax[1].set_ylabel("signal"); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:

# ---- fib-sphere trajectory: spoke ENDPOINTS only -----------------------------
# The kooshball trajectory is one centered radial spoke per echo (25*384 = 9600
# spokes); the spoke directions tile the sphere in golden-angle (Fibonacci)
# order.  The usual visualization draws every spoke as a line from the origin
# to the ball surface -> a 9600-line hairball.  Here each spoke is reduced to
# its far endpoint (outermost ADC sample, normalized onto the unit sphere), so
# only the fib-sphere point cloud is shown - no spokes.
K3 = k_full.reshape(3, N_segments, N_readout, N_x).cpu().numpy()   # (3, S, P, J)
r  = np.linalg.norm(K3, axis=0)                                    # (S, P, J)
j_edge = r.argmax(axis=-1)                                         # outermost sample per spoke
si, pi = np.meshgrid(np.arange(N_segments), np.arange(N_readout), indexing='ij')
tips = K3[:, si.ravel(), pi.ravel(), j_edge.ravel()]               # (3, S*P)
endpoints = tips / np.linalg.norm(tips, axis=0, keepdims=True)     # onto unit sphere
# (readouts are centered, so the antipodal tip is sampled too - to show both
#  endpoints of every diameter, use np.vstack([endpoints, -endpoints]))

order = np.arange(endpoints.shape[1])   # temporal = golden-angle spoke order
fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection='3d')
sc = ax.scatter(endpoints[0], endpoints[1], endpoints[2],
                c=order, cmap='viridis', s=1.5, linewidths=0)
fig.colorbar(sc, ax=ax, shrink=0.6, label='spoke (echo) order')
ax.set_box_aspect((1, 1, 1))
ax.set_xlabel('kx'); ax.set_ylabel('ky'); ax.set_zlabel('kz')
ax.set_title(f'fib-sphere trajectory - {endpoints.shape[0]} spoke endpoints (no spokes)')
plt.show()


## 6. kwic butterfly (Option A: per-echo-index temporal windows)


In [ ]:
# ---- kwic view-sharing (Option A: per-echo-index temporal windows) ----------
# Ported 1:1 from the 2D IR_TrueFISP_bssfp.ipynb butterfly, mapped onto the
# 3D per-echo-index layout:
#   * frame p is built from echo index p of EVERY segment (25 spokes for the
#     center ring), each frame sitting at a single effective TI = TI_START + p*P;
#   * for radial sample jj, the window of echo indices centered on p is
#     ring[jj] wide, with ring[jj] a Fibonacci count that grows from N_center
#     (k-space center) to N_max (periphery) - the golden-ratio view-sharing
#     filter of Ehses et al.;
#   * windows clip at the segment boundaries (early frames keep their center,
#     which conditions S0 in the fit);
#   * the gather is RAW (no per-echo dephasing), matching the 2D reference:
#     the 180-deg bSSFP cycle's per-echo sign is removed by the per-voxel
#     steady-state phase reference in the transient cell, not per-sample.
FIB = [8, 13, 21, 34, 55, 89, 144, 233, 377, 610]
def _next_fib(x):
    return next(f for f in FIB if f >= x)

# NOTE on CG stability (measured, not assumed): the CG system is ill-conditioned
# at BOTH sample counts - CG gives a clean image up to ~20 iters and then
# diverges into dark ringed garbage at 60-120 iters, regardless of N_max
# (ridge terms don't help - they blow up).  So the CG-20 image IS the good one
# and EVERY CG in this notebook is capped at ~20 iters: the DeepADMM training
# uses early-iteration CG (4/8/12) as the "aliased" inputs and CG-20 as the
# reference - no high-iteration CG anywhere.

N_center = 8      # echoes for the k-space center
N_max    = 55     # outermost Fibonacci ring (= time window at the periphery)
frame_advance = 4 # echo-index advance per frame (8 * 12.65 ms = ~101 ms)

# radius of each radial sample (0 at center -> 1 at the edge) from the actual
# trajectory, and its Fibonacci ring count
rad = k_full.reshape(3, N_segments, N_readout, N_x)[:, 0, 0, :].norm(dim=0)
r_frac = rad / rad.max()
ring = np.array([_next_fib(max(N_center, int(np.ceil(N_max * float(r))))) for r in r_frac])
print(f"ring: center={ring.min()}, edge={ring.max()}, sum={ring.sum()}")

# butterfly offsets: for radial sample jj, the ring[jj] echo indices centered on 0
djs, iis = [], []
for jj in range(N_x):
    lo = -(int(ring[jj]) // 2)
    djs.extend(range(lo, lo + int(ring[jj])))
    iis.extend([jj] * int(ring[jj]))
dj = np.asarray(djs, dtype=np.int64)     # (Kb,) echo-index offset
ii = np.asarray(iis, dtype=np.int64)     # (Kb,) radial sample index
Kb = dj.size

# frame centers (echo index), windows clipped at the boundaries
centers = list(range(0, N_readout, frame_advance))          # 0, 8, ..., 376
n_frames = len(centers)
ti_frames = np.array([TI_START + c * P_echo for c in centers])
print(f"frames: {n_frames}, TI span {ti_frames[0]*1e3:.0f} - {ti_frames[-1]*1e3:.0f} ms")

# Per-sample weight.  The 2D reference gathers the butterfly RAW (no per-echo
# dephasing) and references each frame to the steady-state phase per voxel,
# which removes the global (−1)^e sign alternation of the 180-deg cycle.  We
# default to that (DEPHASE=False).  Measured on this data the per-echo phase
# drifts only ~0.6 deg/echo, so the simple (−1)^e dephasing barely helps; the
# transient is carried by the per-voxel SS phase reference instead.
DEPHASE = False
seg_off = (np.arange(N_segments)[:, None] * (N_readout * N_x))  # (S, 1)
keep_np = keep_full.cpu().numpy()
kwic_idx, kwic_w = [], []
for c in centers:
    e = np.clip(c + dj, 0, N_readout - 1)                  # (Kb,) echo index
    flat = (seg_off + (e[None, :] * N_x + ii[None, :])).ravel()   # (S*Kb,)
    flat = flat[np.asarray(keep_np[flat], dtype=bool)]     # drop |k| > pi
    if DEPHASE:
        w = np.where(((flat // N_x) % N_readout) % 2 == 0, 1.0, -1.0).astype(np.float32)
    else:
        w = np.ones(len(flat), dtype=np.float32)
    kwic_idx.append(torch.as_tensor(flat, device=device))
    kwic_w.append(torch.as_tensor(w, device=device))
K_frame = kwic_idx[0].numel()
print(f"K per frame: {K_frame} (={N_segments} x {Kb} per segment)")

# k-space for the CG operator (cycles/voxel)
kwic_k = [k_norm[:, idx] for idx in kwic_idx]


## 7. Normal-NUFFT reconstruction of all kwic frames


In [ ]:
# ---- reconstruction of all kwic frames --------------------------------------
# Two reconstructions per frame:
#   (a) normal NUFFT preview: density-compensated adjoint + adaptive combine
#       (fast, good enough for the transient preview);
#   (b) CG-SENSE: iterative least squares with the exact 3D nonuniform
#       Fourier operator (no DCF needed - the sampling is modeled exactly).
#
# Timing on the A100: (a) ~0.2 s/frame, (b) ~20 s/frame (20 iters, K~56k)
# -> all 48 frames take ~2 min (a) + ~16 min (b).

class NUFFT3DOperator(LinearOperator):
    'F: single-coil image (nX,nY,nZ) -> multi-coil k-space (nC, nK).  k: (3, nK) in cycles/voxel.'
    def __init__(self, k, input_shape, device=None):
        nD, nK = k.shape
        nC, nX, nY, nZ = input_shape
        self.n_modes = (nX, nY, nZ)
        self.k = (-k).unsqueeze(-1)             # (3, nK, 1); sign convention of the module
        self.forward_shape = (nC, nX, nY, nZ, 1)
        self.adjoint_shape = (nC, nK)
        self.shape = (
            int(2 * torch.prod(torch.tensor(self.adjoint_shape))),
            int(2 * torch.prod(torch.tensor(self.forward_shape))),
        )
        self.nufft_ob = KbNufft(im_size=self.n_modes).to(device)
        self.adj_ob = KbNufftAdjoint(im_size=self.n_modes).to(device)
        self.dtype = torch.float32
        self.device = device

    def _matvec(self, x):
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.forward_shape)
        y = nonuniform_fourier_transform_forward(self.k, x, nufft_ob=self.nufft_ob)
        return torch.view_as_real(y.reshape(-1)).reshape(-1)

    def _rmatvec(self, x):
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.adjoint_shape)
        y = nonuniform_fourier_transform_adjoint(self.k, x, self.n_modes, adj_ob=self.adj_ob)
        return torch.view_as_real(y.reshape(-1)).reshape(-1)

C_op = ChannelOperator(coil_c64, (n, n, n))

def recon_nufft(f):
    'normal NUFFT preview of kwic frame f: (n, n, n) complex'
    idx, w = kwic_idx[f], kwic_w[f]
    s = signal[:, idx] * w[None, :]
    return combine(nufft_recon(s, k_full[:, idx], n))

def recon_cg(f, maxiter=20, x0=None):
    """CG-SENSE of kwic frame f -> (n, n, n) complex.

    IMPORTANT: keep maxiter <= ~20.  The conjugate_gradient in
    Modules/NumSolvers.py is only trustworthy in its first ~20
    iterations (g_prev is initialized to ones, and the ill-conditioned
    gridding NUFFT system diverges past ~30 iters into dark ringed
    garbage - measured at both K=22.8k and K=56k; ridge terms do not
    help).  CG-20 is the good, stable image."""
    idx = kwic_idx[f]
    F = NUFFT3DOperator(kwic_k[f], (N_channels, n, n, n))
    Q = C_op.H @ F.H @ F @ C_op
    d = (signal[:, idx] * kwic_w[f][None, :]).contiguous().view(torch.float32).ravel()
    b = C_op.H @ F.H @ d
    x = torch.zeros(n**3, dtype=torch.complex64).view(torch.float32) if x0 is None else x0
    x = conjugate_gradient(Q, b, x, maxiter=maxiter, disable_progressbar=True)
    return x.view(torch.complex64).reshape(n, n, n)

t1 = time.time()
frames_nufft = {}
for f in range(n_frames):
    frames_nufft[f] = recon_nufft(f)
    print(f"frame {f:2d}/{n_frames-1}  TI = {ti_frames[f]*1e3:6.0f} ms   (NUFFT {time.time()-t1:.0f} s)")
print(f"all NUFFT previews: {time.time()-t1:.0f} s")


## 8. CG-SENSE reconstruction of all kwic frames (long)


In [ ]:
# ---- CG-SENSE reconstruction of all kwic frames (the long part) -------------
# ~20 s/frame on the A100 (20 iters, K ~ 56k) -> ~16 min for all 48 frames.
# Skip by setting RUN_CG = False (the fit cells then use the NUFFT previews).
RUN_CG = True
CG_ITERS = 20

# which frames to reconstruct (subset for quick tests; use list(range(n_frames))
# for the full run).  Every downstream cell iterates over CG_FRAMES, so a
# reduced subset stays consistent end to end.
CG_FRAMES = list(range(n_frames - 65))

if RUN_CG:
    t1 = time.time()
    frames_cg = {}
    for f in tqdm(CG_FRAMES, desc="CG-SENSE"):
        frames_cg[f] = recon_cg(f, maxiter=CG_ITERS)
        if f % 8 == 0:
            print(f"  frame {f:2d}  TI = {ti_frames[f]*1e3:6.0f} ms   ({time.time()-t1:.0f} s)")
    print(f"all CG frames ({len(CG_FRAMES)}): {time.time()-t1:.0f} s")
    frames_fit = {f: frames_cg[f] for f in CG_FRAMES}
else:
    frames_fit = {f: frames_nufft[f] for f in CG_FRAMES}

print(f"using {'CG' if RUN_CG else 'NUFFT'} reconstructions for quantification ({len(CG_FRAMES)} frames)")





## 9. DeepADMM: learned aliasing denoiser (sharpens the CG frames)


In [ ]:

# ---- fixed DeepADMM (normalized data term) + constants -----------------------
# min  (1/2) ||A x - d||^2 / nK  +  rho * g(x),   g = learned aliasing denoiser.
#
# Q and b passed to forward() MUST be the *normalized* data term:
#     Q = A.H @ A / nK ,   b = A.H @ d / nK        (nK = # complex samples)
# For kwic frames lambda_max(Q) ~ 1e-3, so rho ~ 3e-4 is on the same scale and
# the regularizer actually has an effect.  (With the unnormalized A.H@A,
# lambda_max ~ 450 and rho=0.1 is ~2e-4 of the data term: the fixed point is
# numerically the plain-CG solution and the "learned" model cannot move the
# image at all.)

from Modules.Regularizer import Regularizer
from Modules.Linops import IdentityOperator

class DeepADMM(nn.Module):
    def __init__(self, shape, rho_init=3e-4, maxiter_admm=1, maxiter_cg=20,
                 kernel_size=(3, 3, 3), features=16, num_of_resblocks=4,
                 scale_factor=1, checkpoint_iterations=True):
        super().__init__()
        self.rho = nn.Parameter(torch.tensor(rho_init, dtype=torch.float32))
        self.regularizer = Regularizer(
            shape, kernel_size=kernel_size, features=features,
            num_of_resblocks=num_of_resblocks, scale_factor=scale_factor,
        )
        self.I = IdentityOperator(shape)
        self.maxiter_admm = maxiter_admm
        self.maxiter_cg = maxiter_cg
        self.checkpoint_iterations = checkpoint_iterations

    def _step(self, x, z, u, Q, b, maxiter_cg):
        x = conjugate_gradient(Q + self.rho * self.I, b=b + self.rho * (z - u),
                               x=x, maxiter=maxiter_cg, disable_progressbar=True)
        z = self.regularizer(x + u)
        u = u + x - z
        return x, z, u

    def forward(self, Q, b, x, z=None, maxiter_admm=None, maxiter_cg=None,
                disable_progressbar=True):
        # warm start: x = data-fitting solution (the CG result), z = g(x)
        n_admm = self.maxiter_admm if maxiter_admm is None else maxiter_admm
        n_cg = self.maxiter_cg if maxiter_cg is None else maxiter_cg
        if z is None:
            z = self.regularizer(x)
        u = torch.zeros(self.I.shape[0], dtype=x.dtype)
        use_ckpt = self.checkpoint_iterations and self.training and torch.is_grad_enabled()
        for _ in range(n_admm):
            if use_ckpt:
                x, z, u = checkpoint(self._step, x, z, u, Q, b, n_cg, use_reentrant=False)
            else:
                x, z, u = self._step(x, z, u, Q, b, n_cg)
        return x

from torch.utils.checkpoint import checkpoint

# ---- hyperparameters ---------------------------------------------------------
ADMM_CHECKPOINT = "deepadmm_kwic_bssfp.pt"
LOAD_CHECKPOINT = True     # if the file exists, skip pair generation + pretrain
ADMM_RHO = 3e-4            # fixed (a learnable rho collapses to ~0)
# TRAINING DESIGN (early-iteration pairs - no high-iteration CG anywhere):
#   reference  = CG-20 of the FULL frame      (clean, stable at all kwic settings)
#   aliased    = CG-4 / CG-8 / CG-12 of the FULL frame
# The conjugate_gradient in Modules/NumSolvers.py is only trustworthy in its
# first ~20 iterations: g_prev is initialized to ones (the first CG step is
# wrong) and the ill-conditioned gridding NUFFT system diverges past ~30
# iters (measured: clean at 20, dark ringed garbage at 60-120, at BOTH
# K=22.8k and K=56k; ridge terms don't help - they blow up).  So every CG in
# this pipeline is early-stopped.
REF_ITERS  = 20            # CG iters for the references
CG_WARM    = 20            # CG iters for the inference warm start
ADMM_ITERS = 20            # inner CG iters per ADMM step at inference
REF_FRAMES = [4, 16, 28]   # kwic frames used as references (early -> late TI)
SUB_ITERS  = (4, 8, 12, 16)    # aliased = these early CG-iteration counts


In [ ]:

# ---- train the denoiser as an aliasing denoiser ------------------------------
# Pseudo-GT : full kwic frame reconstructed with CG-20 (clean, stable).
# Aliased   : the SAME frame with fewer CG iterations (4/8/12/16) - in
#             exactly the domain the denoiser sees at inference.
# Loss = MSE(denoise aliased) + 0.5 * MSE(identity on clean) so the net is
# ~identity on already-clean images (the behavior of a genuine prox).
#
# Cost on the A100: references ~1.5 min, pairs ~3 min, pretrain ~40 min (one
# time; LOAD_CHECKPOINT skips all of it afterwards).
# IMPORTANT: delete ADMM_CHECKPOINT before the first run - the existing file
# was trained with diverged high-iteration references and is useless.

if LOAD_CHECKPOINT and os.path.exists(ADMM_CHECKPOINT):
    # NOTE: any checkpoint trained with high-iteration (30+) CG references is
    # BAD - those references diverged to dark ringed garbage.  Delete
    # ADMM_CHECKPOINT and re-run this cell if in doubt.
    torch.manual_seed(0)
    model = DeepADMM((n, n, n, 1), rho_init=ADMM_RHO, maxiter_admm=1,
                     maxiter_cg=ADMM_ITERS)
    model.regularizer.regularizer.load_state_dict(
        torch.load(ADMM_CHECKPOINT, map_location="cuda")["denoiser"])
    model.rho.requires_grad_(False)
    print(f"loaded denoiser from {ADMM_CHECKPOINT}")
else:
    # ---- full-frame references (CG-20: clean and stable) ----------------------
    print(f"computing full-frame references (CG {REF_ITERS} iters)...")
    ref_vols = {}
    for f in REF_FRAMES:
        x = recon_cg(f, maxiter=REF_ITERS)
        ref_vols[f] = x
        print(f"  frame {f:2d} (TI = {ti_frames[f]*1e3:.0f} ms) done")

    # ---- aliased training pairs: early-iteration CG of the SAME full frame ----
    # Same operator and samples as the references - only the CG iteration
    # count differs (4/8/12 vs 20), so the aliased images sit in exactly the
    # same domain as the CG-20 warm starts the denoiser sees at inference,
    # and nothing here ever runs CG past ~20 iters.
    pairs_y, pairs_x = [], []
    for f in REF_FRAMES:
        for nit in SUB_ITERS:
            y = recon_cg(f, maxiter=nit)
            pairs_y.append(y)
            pairs_x.append(ref_vols[f])
            print(f"    frame {f:2d} CG-{nit}: done", flush=True)
    Y = torch.stack(pairs_y)[:, None]      # (N, 1, n, n, n) aliased CG reconstructions
    X = torch.stack(pairs_x)[:, None]      # (N, 1, n, n, n) full-frame references
    print(f"pairs: {tuple(Y.shape)}")

    # ---- pretrain the ResNet denoiser -----------------------------------------
    # n=128 note: activations scale as n^3 (8x vs n=64), and the original
    # step backpropagated TWO full forward graphs at B=6 (aliased term +
    # identity term in one combined loss) -> ~30-40 GB of activations and an
    # OOM on a 40 GB card.  Training math below is IDENTICAL (gradients from
    # both sub-losses are accumulated and one opt.step() is taken per step),
    # but peak activation memory is cut from ~30 GB to a few GB by:
    #   * B = 2 instead of 6
    #   * backpropagating the two sub-losses separately (one graph alive)
    #   * per-resblock activation checkpointing (ResNetCheckpoints=True)
    from Modules.Resnet import ResNet
    denoiser = ResNet(contrasts=1, features=16, num_of_resblocks=4, scale_factor=1,
                      kernel_size=(3, 3, 3), device=device, dtype=torch.complex64,
                      ResNetCheckpoints=True)
    opt = torch.optim.Adam(denoiser.parameters(), lr=1e-3)
    B = 2
    losses = []
    for phase, (n_steps, lr) in enumerate([(1200, 1e-3), (4000, 3e-4)]):
        for pg in opt.param_groups:
            pg["lr"] = lr
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_steps) if phase else None
        for step in tqdm(range(n_steps), desc=f"denoiser pretrain phase {phase+1}"):
            sel = torch.randperm(len(Y))[:B]
            opt.zero_grad()
            # 1) aliasing term - own forward graph, backpropagated right away
            pred = denoiser(Y[sel])
            l1 = ((pred - X[sel]).abs() ** 2).mean()
            l1.backward()
            del pred
            # 2) identity term - separate graph (only one alive at a time)
            pred = denoiser(X[sel])
            l2 = 0.5 * ((pred - X[sel]).abs() ** 2).mean()
            l2.backward()
            losses.append(float(l1.item() + l2.item()))
            del pred, l1, l2
            opt.step()
            if sched is not None:
                sched.step()
    print(f"final pretrain loss {losses[-1]:.3e}")

    # ---- assemble the DeepADMM model -------------------------------------------
    torch.manual_seed(0)
    model = DeepADMM((n, n, n, 1), rho_init=ADMM_RHO, maxiter_admm=1,
                     maxiter_cg=ADMM_ITERS)
    model.regularizer.regularizer.load_state_dict(denoiser.state_dict())
    model.rho.requires_grad_(False)
    # sanity check: the denoiser must be ~identity on a clean CG image
    with torch.no_grad():
        x_chk = denoiser(ref_vols[REF_FRAMES[0]].reshape(1, n, n, n)).reshape(n, n, n)
    err_after = torch.abs(x_chk - ref_vols[REF_FRAMES[0]]).mean() / ref_vols[REF_FRAMES[0]].abs().mean()
    print(f"identity check on clean image: rel change = {float(err_after):.4f} (should be small)")
    torch.save({"denoiser": denoiser.state_dict(),
                "losses": losses}, ADMM_CHECKPOINT)
    print(f"model saved to {ADMM_CHECKPOINT}")

model.eval()

def recon_admm(f, x0, maxiter_admm=1, maxiter_cg=ADMM_ITERS):
    'DeepADMM on kwic frame f, warm-started from the CG result x0.'
    idx = kwic_idx[f]
    F_p = NUFFT3DOperator(kwic_k[f], (N_channels, n, n, n))
    d_p = (signal[:, idx] * kwic_w[f][None, :]).contiguous().view(torch.float32).ravel()
    nK  = d_p.numel() // 2
    Q_p = (C_op.H @ F_p.H @ F_p @ C_op) * (1.0 / nK)    # normalized data term
    b_p = (C_op.H @ F_p.H @ d_p) * (1.0 / nK)
    x0f = x0.reshape(-1).view(torch.float32).reshape(-1)
    with torch.no_grad():
        x = model(Q_p, b_p, x0f, maxiter_admm=maxiter_admm, maxiter_cg=maxiter_cg)
    return x.view(torch.complex64).reshape(n, n, n)


In [ ]:

# ---- DeepADMM inference: all kwic frames (warm-started from CG) --------------
# ~ (CG warm start + one ADMM step with inner CG) per frame -> ~40-60 s/frame
# on the A100, i.e. ~30-45 min for all 48 frames.
t1 = time.time()
frames_admm = {}
for f in tqdm(CG_FRAMES, desc="DeepADMM"):
    x_cg = recon_cg(f, maxiter=CG_WARM)
    frames_admm[f] = recon_admm(f, x_cg)
    if f % 8 == 0:
        print(f"  frame {f:2d}  TI = {ti_frames[f]*1e3:6.0f} ms   ({time.time()-t1:.0f} s)")
print(f"all DeepADMM frames ({len(CG_FRAMES)}): {time.time()-t1:.0f} s")

# replace the frames used downstream (transient + quantification)
frames_fit = frames_admm
print("frames_fit <- DeepADMM reconstructions")

# quick visual: CG vs DeepADMM at a few TIs (only frames that were computed)
mid = n // 2
cmp_frames = [f for f in [4, 16, 28, CG_FRAMES[-1]] if f in frames_admm][-4:]
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, f in enumerate(cmp_frames):
    for row, (fn, lab) in enumerate([(lambda g: recon_cg(g, maxiter=CG_WARM), "CG"),
                                     (lambda g: frames_admm[g], "DeepADMM")]):
        a = fn(f).abs().cpu().numpy()
        axes[row, i].imshow(np.rot90(a[:, :, mid].T, 2), cmap="gray",
                            vmin=0, vmax=np.percentile(a, 99.9))
        axes[row, i].set_title(f"{lab}, TI={ti_frames[f]*1e3:.0f} ms", fontsize=9)
        axes[row, i].axis("off")
plt.tight_layout()
plt.show()


# ## 9.5. ContrastADMM: cross-TI denoising via the Ehses contrast manifold

In [ ]:

# ---- ContrastADMM v2: contrast-manifold denoising (stabilized) ---------------
# v1 (hard-projection ADMM) diverged - two bugs:
#   1) sse = ||y||^2 - w^H A^H y suffers catastrophic cancellation in float32
#      (both terms ~ ||y||^2, the difference is the small true residual).
#      Where A = [a, c] is ill-conditioned (large T1*) the difference goes
#      spuriously NEGATIVE, so the per-voxel T1* argmin snapped to roundoff-
#      driven grid values -> per-voxel T1* jumps -> speckled z and maps.
#   2) with a non-convex projection, the ADMM dual u accumulated the
#      systematic |x - z| ~ O(|x|) mismatch every round; once rho*(z - u)
#      became comparable to the data term, the loop fed its own noise back
#      into the fit (S0/Sss drifted to ~7-10) and ran away - the residual
#      plateaued at ~1.1 instead of decaying.
# v2 keeps the physics, changes three things:
#   * sse = DIRECT residual ||y - A w||^2 (a sum of squares, no cancellation),
#     2x2 solves in float64;
#   * T1* refined by a parabola through the 3 SSE points at the argmin
#     (sub-50 ms resolution, continuous across rounds - no argmin jumps);
#   * no dual variable: the u-free contraction iteration
#         X <- CG-solve  (Q_f + rho I) x = b_f + rho * P(X)_f
#     which cannot run away.  Final Z = P(X): on-manifold frames + maps.
# Cost: ~2 min operator build + ~N rounds x (48 frames x 10 CG iters + 1 fit)
# ~ 3-4 min/round.  Quick look: CTR_ITERS = 4.

from Modules.Linops import IdentityOperator

CTR_RHO        = ADMM_RHO   # 3e-4: same scale as lambda_max(Q) ~ 1e-3
CTR_ITERS      = 8          # fixed-point rounds
CTR_CG_ITERS   = 10         # inner CG iters per x-update (warm-started, <=20 cap)
CTR_T1STAR_GRID = np.arange(0.05, 5.0, 0.05)

ctr_frames = list(CG_FRAMES)
T_fit = np.array([ti_frames[f] for f in ctr_frames])          # (F,) effective TI (s)

# ---- initialize + initial projection (fast health check before the build) ----
tc = time.time()
_frames_cg = globals().get("frames_cg", {})
X = torch.stack([_frames_cg.get(f, frames_fit[f]) for f in ctr_frames])   # (F, n, n, n)

# ---- the manifold projector P (v2): per-voxel Ehses fit, stabilized ----------
# Model per voxel:  y_f ~ e^{i phi} (Sss a_f + S0 c_f),  a_f = 1 - E_f, c_f = -E_f,
# E_f = exp(-t_f/T1*).  For fixed T1*: exact complex LS in
# (C1, C2) = (Sss e^{i phi}, S0 e^{i phi}):  w = (A^H A)^{-1} A^H y, A = [a, c].
# v2: direct-residual SSE (no float32 cancellation), float64 grid solves,
# parabolic T1* refinement.  y: (F, n, n, n) complex -> (z, params, good).
def contrast_project(y):
    Fv = y.shape[0]
    yv = y.reshape(Fv, -1)                                   # (F, nvox) complex64
    nvox = yv.shape[1]
    ngrid = len(CTR_T1STAR_GRID)
    yv128 = yv.to(torch.complex128)
    sse_mat = torch.empty((ngrid, nvox), device=y.device, dtype=torch.float32)
    for i, T1s in enumerate(CTR_T1STAR_GRID):
        E = torch.as_tensor(np.exp(-T_fit / T1s), dtype=torch.float64, device=y.device)
        a, c = 1.0 - E, -E                                   # (F,) float64
        c2 = torch.stack([(a[:, None] * yv128).sum(0), (c[:, None] * yv128).sum(0)], dim=0)
        G2 = torch.tensor([[float((a * a).sum()), float((a * c).sum())],
                           [float((a * c).sum()), float((c * c).sum())]],
                          dtype=torch.complex128, device=y.device)
        w = torch.linalg.solve(G2, c2)                       # (2, nvox) complex128
        Aw = a[:, None] * w[0][None, :] + c[:, None] * w[1][None, :]     # (F, nvox)
        sse_mat[i] = ((yv128 - Aw).abs() ** 2).sum(0).float()            # DIRECT
    best_i = sse_mat.argmin(0)                                # (nvox,)
    # parabolic T1* refinement through (i-1, i, i+1)
    h = float(CTR_T1STAR_GRID[1] - CTR_T1STAR_GRID[0])
    iL = (best_i - 1).clamp(min=0)
    iR = (best_i + 1).clamp(max=ngrid - 1)
    idx = torch.arange(nvox, device=y.device)
    yL, yC, yR = sse_mat[iL, idx], sse_mat[best_i, idx], sse_mat[iR, idx]   # (nvox,)
    den = yL - 2.0 * yC + yR
    scale = (yL + yC + yR).abs().clamp(min=1e-30)
    delta = torch.where(den.abs() > 1e-9 * scale,
                        0.5 * h * (yL - yR) / den.clamp(min=1e-30),
                        torch.zeros((), device=y.device))
    delta = delta.clamp(-h, h)
    T1ref = torch.as_tensor(CTR_T1STAR_GRID, dtype=torch.float64,
                            device=y.device)[best_i] + delta
    del sse_mat
    # exact complex LS at the (continuous, per-voxel) T1ref
    Tfit32 = torch.as_tensor(T_fit, dtype=torch.float32, device=y.device)
    Eref = torch.exp(-Tfit32[:, None] / T1ref.to(torch.float32)[None, :])   # (F, nvox)
    Aref = torch.stack([1.0 - Eref, -Eref], dim=0)                         # (2, F, nvox)
    Arefc = Aref.to(torch.complex64)
    c2 = torch.einsum('jfv,fv->jv', Arefc, yv).permute(1, 0).unsqueeze(-1) # (nvox, 2, 1)
    G = torch.einsum('jfv,kfv->jkv', Aref, Aref).permute(2, 0, 1).contiguous()
    w = torch.linalg.solve(G.to(torch.complex64), c2)                 # (nvox, 2, 1)
    C1, C2 = w[:, 0, 0], w[:, 1, 0]
    del yv128, Aref, Arefc, G, c2, w
    Sss = C1.abs()
    S0  = C2.abs()
    ephi = (C1 / C1.abs().clamp(min=1e-30) + C2 / C2.abs().clamp(min=1e-30)) / 2
    ephi = ephi / ephi.abs().clamp(min=1e-30)
    bad = (Sss < 1e-9) | (S0 < 1e-9)
    if bool(bad.any()):
        ephi = torch.where(bad, torch.complex(1.0, 0.0), ephi)
    z = ((Sss[None, :] * (1.0 - Eref) + S0[None, :] * (-Eref)) * ephi[None, :])
    z = z.reshape(Fv, n, n, n)
    params = {
        "T1star": T1ref.float().reshape(n, n, n),
        "Sss":    Sss.reshape(n, n, n),
        "S0":     S0.reshape(n, n, n),
        "phi":    torch.angle(ephi).reshape(n, n, n),
    }
    eps = 1e-3 * float(Sss.max())
    good = ((Sss > eps) & (S0 > eps)).reshape(n, n, n)
    return z, params, good

Z, Pn, good = contrast_project(X)
T1i = Pn["T1star"][good].cpu().numpy()
d0 = float(((X - Z).abs() / X.abs().clamp(min=1e-12)).mean())
print(f"init P(x0): {time.time()-tc:.0f} s; sensible voxels: {good.float().mean():.1%}")
print(f"init |x - P(x)|/|x| = {d0:.3f}   (small ~ the CG-20 frames already sit on the curve)")
print(f"init T1* p10/50/90 (s) = {np.percentile(T1i, [10, 50, 90]).round(3)}   "
      f"(healthy: narrow, tissue-like - NOT spread over the whole 0.05-5.0 grid)")

# ---- per-frame data terms (built once, reused every round) -------------------
tc = time.time()
print("building per-frame data terms (Q_f, b_f)...")
ops = {}
for f in ctr_frames:
    idx = kwic_idx[f]
    F_p = NUFFT3DOperator(kwic_k[f], (N_channels, n, n, n))
    d_p = (signal[:, idx] * kwic_w[f][None, :]).contiguous().view(torch.float32).ravel()
    nK  = d_p.numel() // 2
    ops[f] = ((C_op.H @ F_p.H @ F_p @ C_op) * (1.0 / nK),
              (C_op.H @ F_p.H @ d_p) * (1.0 / nK))
print(f"operators for {len(ops)} frames ready ({time.time()-tc:.0f} s)")

# ---- u-free fixed-point iteration (a contraction - cannot run away) ----------
I_op = IdentityOperator((n, n, n, 1))
traj = []
tc = time.time()
for it in range(CTR_ITERS):
    # x-update: per-frame warm-started CG on (Q_f + rho I) toward (b_f + rho Z_f)
    for j, f in enumerate(tqdm(ctr_frames, desc=f"ContrastADMM v2 round {it+1}/{CTR_ITERS}")):
        Q_f, b_f = ops[f]
        rhs = b_f + CTR_RHO * Z[j].reshape(-1).view(torch.float32)
        xv = X[j].reshape(-1).view(torch.float32)
        xv = conjugate_gradient(Q_f + CTR_RHO * I_op, b=rhs, x=xv,
                                maxiter=CTR_CG_ITERS, disable_progressbar=True)
        X[j] = xv.view(torch.complex64).reshape(n, n, n)
    # projection (this round's target for the next x-update)
    Z, Pn, good = contrast_project(X)
    d = float(((X - Z).abs() / X.abs().clamp(min=1e-12)).mean())
    traj.append(d)
    T1r = Pn["T1star"][good].cpu().numpy()
    print(f"round {it+1}/{CTR_ITERS}: |X - P(X)|/|X| = {d:.3f}   "
          f"T1* p10/50/90 = {np.percentile(T1r, [10, 50, 90]).round(3)}   ({time.time()-tc:.0f} s)")

# ---- outputs: soft frames (data + manifold), on-manifold frames, maps --------
frames_soft       = {f: X[j] for j, f in enumerate(ctr_frames)}   # the compromise
frames_onmanifold = {f: Z[j] for j, f in enumerate(ctr_frames)}   # exactly on the curve
frames_contrast = frames_soft                                     # what downstream uses

# Ehses Eqs. 2-4 (same conversion as the section-11 fit)
c_half, s_half = np.cos(alpha / 2), np.sin(alpha / 2)
T1star = Pn["T1star"].cpu().numpy()
Sss = Pn["Sss"].cpu().numpy()
S0  = Pn["S0"].cpu().numpy()
good3 = good.cpu().numpy().astype(bool)
T1_c = np.where(good3, T1star * (S0 / np.where(good3, Sss, 1.0)) * c_half, np.nan)
T2_c = np.where(good3, T1star * s_half**2 / (1.0 - (Sss / np.where(good3, S0, 1.0)) * c_half), np.nan)
M0_c = np.where(good3, S0 / s_half, np.nan)
PD_scale = float(np.nanmean(M0_c[masks["CSF"] & good3]))
PD_c = M0_c / PD_scale
print(f"fit: {good3.mean()*100:.1f}% voxels sensible; CSF M0 normalized to PD 1.0")
print("\nContrastADMM v2 maps vs phantom GT (k-space fit):")
print(f"{'tissue':6s} {'T1':>8s} {'T1 GT':>7s}  {'T2':>8s} {'T2 GT':>7s}  {'PD':>7s} {'PD GT':>7s}")
for t in NAMES:
    g = masks[t] & good3
    gt = TISSUES[t]
    print(f"{t:6s} {np.nanmean(T1_c[g]):8.3f} {gt['T1']:7.2f}  "
          f"{np.nanmean(T2_c[g]):8.3f} {gt['T2']:7.2f}  "
          f"{np.nanmean(PD_c[g]):7.3f} {gt['PD']:7.2f}")

# ---- frame comparison: CG-20 / soft compromise / on-manifold ------------------
mid = n // 2
pick = [ctr_frames[0], ctr_frames[len(ctr_frames)//3],
        ctr_frames[2*len(ctr_frames)//3], ctr_frames[-1]]
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for i, f in enumerate(pick):
    for row, (fn, lab) in enumerate([(lambda g: _frames_cg.get(g, frames_fit[g]), "CG-20"),
                                     (lambda g: frames_soft[g], "ContrastADMM (soft)"),
                                     (lambda g: frames_onmanifold[g], "on-manifold P(X)")]):
        a = fn(f).abs().cpu().numpy()
        axes[row, i].imshow(np.rot90(a[:, :, mid].T, 2), cmap="gray",
                            vmin=0, vmax=np.percentile(a, 99.9))
        axes[row, i].set_title(f"{lab}, TI={ti_frames[f]*1e3:.0f} ms", fontsize=9)
        axes[row, i].axis("off")
plt.suptitle("ContrastADMM v2: CG-20 / soft compromise / on-manifold")
plt.tight_layout()
plt.show()

# ---- maps + convergence --------------------------------------------------------
vmax1 = np.nanpercentile(T1_c[brain_mask], 98)
vmax2 = np.nanpercentile(T2_c[brain_mask], 98)
vmaxp = np.nanpercentile(PD_c[brain_mask], 98)
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (M, vmax, title, cmap) in zip(axes, [
    (T1_c, vmax1, "T1 (s), mid slice", "hot"),
    (T2_c, vmax2, "T2 (s), mid slice", "hot"),
    (PD_c, vmaxp, "PD (rel., CSF=1), mid slice", "gray"),
]):
    img = np.rot90(np.nan_to_num(M[:, :, mid], nan=0.0).T, 2)
    ax.imshow(img, cmap=cmap, vmin=0, vmax=vmax)
    ax.set_title(title); ax.axis("off")
plt.suptitle("ContrastADMM v2 quantification (maps from the k-space fit)")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.semilogy(traj, "o-")
ax.set_xlabel("round"); ax.set_ylabel("|X - P(X)| / |X|")
ax.set_title("distance of X to the contrast manifold (soft solution plateaus > 0)")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# ---- swap into the downstream pipeline (transient, fit, maps) -------------------
frames_fit = {f: frames_contrast[f] for f in ctr_frames}
print("frames_fit <- ContrastADMM v2 (soft) reconstructions")
print("re-run sections 10-12 for the transient + refit (the refit should land almost")
print("exactly on the parameters above - a self-consistency check on the whole loop)")

## 10. kwic frames + signed tissue-mean transient


In [ ]:
# ---- the contrast evolution: kwic frames (magnitude) -------------------------
mid = n // 2
# shared, robust color scale across all frames: one hot aliased voxel
# must not blow out a frame, and a common vmax makes the intensity growth
# toward steady state visible from frame to frame.
# iterate only the frames that were actually reconstructed (CG_FRAMES);
# pick up to 24 for a 4x6 grid
fit_frames = sorted(frames_fit.keys())
grid_frames = fit_frames[::max(1, len(fit_frames) // 24)][:24]
_slic = np.stack([frames_fit[f].abs().cpu().numpy()[:, :, mid] for f in grid_frames])
#vmax_shared = float(np.percentile(_slic, 95))
fig, axes = plt.subplots(4, 6, figsize=(18, 12))
for i, f in enumerate(grid_frames):
    ax = axes.ravel()[i]
    img = frames_fit[f].abs().cpu().numpy()
    ax.imshow(np.rot90(img[:, :, mid].T, 2), cmap="gray",
              vmin=0)
    ax.set_title(f"TI = {ti_frames[f]*1e3:4.0f} ms", fontsize=8)
    ax.axis("off")
for j in range(len(grid_frames), axes.size):
    axes.ravel()[j].axis("off")
plt.suptitle("kwic bSSFP - one scan, one image per echo-index window (magnitude)")
plt.tight_layout()
plt.show()

# ---- signed tissue-mean transient --------------------------------------------
# Steady-state phase reference (per voxel): reference every frame to a
# steady-state frame.  Use the last FULLY-SAMPLED frame, not the very last
# one - the last kwic frame has a clipped, asymmetric window (its echoes
# run past the end of the segment) so its phase is less reliable, and
# referencing to it makes the final points of the signed series jump.
half_win = N_max // 2
# phase reference: the last COMPUTED frame whose window is not clipped
ref_f = max(f for f in fit_frames if centers[f] + half_win < N_readout)
ref_phase = torch.angle(frames_fit[ref_f])
t_fit = np.array([ti_frames[f] for f in fit_frames])
print(f"phase reference: frame {ref_f} (TI = {ti_frames[ref_f]*1e3:.0f} ms)")

def signed_series(mask):
    m = torch.as_tensor(mask, device=device)
    m = m / m.sum()
    s = [float(torch.real(frames_fit[f] * torch.exp(-1j * ref_phase))[m].sum())
         for f in fit_frames]
    return np.array(s)

def mag_series(mask):
    m = torch.as_tensor(mask, device=device)
    m = m / m.sum()
    return np.array([float(frames_fit[f].abs()[m].sum()) for f in fit_frames])

fig, axes = plt.subplots(1, 2, figsize=(8, 4.5))
for t, col in zip(NAMES, ["tab:blue", "tab:orange", "tab:green"]):
    axes[0].plot(t_fit * 1e3, signed_series(masks[t]), label=t)
    axes[1].plot(t_fit * 1e3, mag_series(masks[t]), label=t)
axes[0].axhline(0, color="k", lw=0.5)
axes[0].set_title("signed tissue-mean IR transient (SS-phase referenced)")
axes[1].set_title("magnitude tissue-mean (unsigned)")
for a in axes:
    a.set_xlabel("TI (ms)"); a.legend(); a.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 11. Ehses per-voxel T1 / T2 / PD fit + ground-truth table


In [ ]:
# ---- Ehses T1 / T2 / PD quantification (per voxel) ---------------------------
# Signal model (Ehses et al. MRM 2013, Eq. 1), on the SIGNED (SS-phase
# referenced) series, with t = absolute effective TI:
#     S(t) = Sss - (S0 + Sss) exp(-t / T1*)
# Closed form: for each T1* candidate the fit is linear in (Sss, S0+Sss)
# (2 x n_frames least squares); T1* is found by a grid scan.
# Then Ehses Eqs. 2-4 (TR << T2 assumed - here P/T2 ~ 0.16 for WM, so T1/T2
# carry a small systematic error):
#     T1 = T1* (S0/Sss) cos(a/2)
#     T2 = T1* sin^2(a/2) / (1 - (Sss/S0) cos(a/2))
#     M0 = S0 / sin(a/2)      (NUFFT scale is arbitrary -> relative PD)
fit_frames = sorted(frames_fit.keys())
S_signed = np.stack([
    torch.real(frames_fit[f] * torch.exp(-1j * ref_phase)).cpu().numpy()
    for f in fit_frames
], axis=0)                                  # (n_fit, n, n, n)
T = np.array([ti_frames[f] for f in fit_frames])   # (n_fit,) absolute TI

T1STAR_GRID = np.arange(0.05, 5.0, 0.05)
nvox = int(np.prod(S_signed.shape[1:]))
Sv = S_signed.reshape(n_frames, -1)         # (n_frames, nvox)
c_half, s_half = np.cos(alpha / 2), np.sin(alpha / 2)

best_sse, best_Sss, best_S0, best_T1s = None, None, None, None
for T1s in T1STAR_GRID:
    E = np.exp(-T / T1s)                     # (n_frames,)
    A = np.stack([1.0 - E, -E], axis=1)      # S = Sss*(1-E) + (S0+Sss)*(-E)
    B = np.linalg.lstsq(A, Sv, rcond=None)[0]        # (2, nvox)
    Sss_c, S0Sss_c = B[0], B[1]
    sse = np.sum((A @ B - Sv) ** 2, axis=0)
    if best_sse is None:
        best_sse, best_Sss, best_S0, best_T1s = sse, Sss_c.copy(), (S0Sss_c - Sss_c).copy(), float(T1s)
    else:
        m = sse < best_sse
        best_sse[m] = sse[m]
        best_Sss[m] = Sss_c[m]
        best_S0[m] = (S0Sss_c - Sss_c)[m]
        best_T1s[m] = float(T1s)

Sss, S0 = best_Sss, best_S0
good = (S0 > 0) & (Sss > 0)                 # physically sensible fit
T1_map = np.where(good, best_T1s * (S0 / Sss) * c_half, np.nan)
T2_map = np.where(good,
                  best_T1s * s_half**2 / (1.0 - (Sss / S0) * c_half),
                  np.nan)
M0_map = np.where(good, S0 / s_half, np.nan)
T1_map, T2_map = T1_map.reshape(n, n, n), T2_map.reshape(n, n, n)
M0_map, good3 = M0_map.reshape(n, n, n), good.reshape(n, n, n)
# The NUFFT amplitude scale is arbitrary, so M0 is a RELATIVE PD.  Normalize
# the CSF (the longest-T1 tissue, cleanest SS signal) to the phantom's PD 1.0:
PD_scale = float(np.nanmean(M0_map[masks["CSF"] & good3]))
PD_map = M0_map / PD_scale
print(f"fit: {good3.mean()*100:.1f}% voxels sensible; CSF M0 normalized to PD 1.0")

# ---- validation against the phantom ground truth -----------------------------
print(f"\n{'tissue':6s} {'T1 fit':>9s} {'T1 GT':>7s}  {'T2 fit':>8s} {'T2 GT':>7s}  {'PD fit':>8s} {'PD GT':>7s}")
for t in NAMES:
    g = masks[t] & good3
    gt = TISSUES[t]
    print(f"{t:6s} {np.nanmean(T1_map[g]):9.3f} {gt['T1']:7.2f}  "
          f"{np.nanmean(T2_map[g]):8.3f} {gt['T2']:7.2f}  "
          f"{np.nanmean(PD_map[g]):8.3f} {gt['PD']:7.2f}")


## 12. T1 / T2 / PD maps


In [ ]:
# ---- T1 / T2 / PD maps -------------------------------------------------------
mid = n // 2
vmax1 = np.nanpercentile(T1_map[brain_mask], 98)
vmax2 = np.nanpercentile(T2_map[brain_mask], 98)
vmaxp = np.nanpercentile(PD_map[brain_mask], 98)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (M, vmax, title, cmap) in zip(axes, [
    (T1_map, vmax1, f"T1 (s), mid slice", "inferno"),
    (T2_map, vmax2, f"T2 (s), mid slice", "viridis"),
    (PD_map, vmaxp, f"PD (rel., CSF=1), mid slice", "plasma"),
]):
    img = np.rot90(np.nan_to_num(M[:, :, mid], nan=0.0).T, 2)
    ax.imshow(img, cmap=cmap, vmin=0, vmax=vmax)
    ax.set_title(title); ax.axis("off")
plt.suptitle("Ehses quantification from 3D kwic bSSFP (one scan)")
plt.tight_layout()
plt.show()

# ---- side-by-side: a few kwic frames + the fitted maps ------------------------
fit_frames = sorted(frames_fit.keys())
pick = [fit_frames[0], fit_frames[len(fit_frames)//3], fit_frames[2*len(fit_frames)//3], fit_frames[-1]]
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, f in enumerate(pick):
    img = frames_fit[f].abs().cpu().numpy()
    axes[0, i].imshow(np.rot90(img[:, :, mid].T, 2), cmap="gray",
                      vmin=0, vmax=vmax_shared)
    axes[0, i].set_title(f"kwic TI={ti_frames[f]*1e3:.0f} ms", fontsize=9)
    axes[0, i].axis("off")
for i, (M, vmax, title) in enumerate([
    (T1_map, vmax1, "T1 (s)"), (T2_map, vmax2, "T2 (s)"), (PD_map, vmaxp, "PD (rel.)")
]):
    img = np.rot90(np.nan_to_num(M[:, :, mid], nan=0.0).T, 2)
    axes[1, i].imshow(img, cmap="inferno", vmin=0, vmax=vmax)
    axes[1, i].set_title(title, fontsize=9); axes[1, i].axis("off")
axes[1, 3].imshow(np.rot90(pd_map.T, 2), cmap="gray", vmin=0, vmax=1.0)
axes[1, 3].set_title("phantom PD (GT)", fontsize=9); axes[1, 3].axis("off")
plt.suptitle("kwic bSSFP frames -> Ehses T1 / T2 / PD maps vs phantom")
plt.tight_layout()
plt.show()


In [ ]:
# =============================================================================
# [NEW] F-as-channels probe -- separate cell; does NOT modify any earlier cell.
#
# Question A: does the stack-ResNet (F frames as channels) plug in with NO
#             change to Modules/Resnet.py?            -> shape/grad smoke test.
# Question B: can channel-mixing LEARN the shared bSSFP curve, self-supervised?
#             Hold out one timepoint, zero its input slot, train the net to
#             predict that frame's measured k-space from the OTHER frames.
#             If the held-out k-space error FALLS, the net is using the other
#             timepoints -> it learned the curve (no Ehses, no ground truth).
#
# NOTE on scaling (v2 -- fixes v1, whose loss was O(100) and unreadable):
#   The notebook's NUFFT pair is a true adjoint pair but NOT unit-norm:
#   F F^H ~= alpha * I on the samples, with
#       alpha ~= (2x oversamp grid)^3 x sum_i |coil_i|^2  ~= 160..260 here.
#   So the zero-filled regrid b_f = C^H F^H d_f is ~alpha x the true frame
#   image.  We therefore feed b_f/alpha_f (the usual aliasing regrid, at
#   true-image scale) as the input; the k-space loss ||F C x_hat - d_f||/||d_f||
#   is then naturally O(1) and the trend is readable.
#
# Assumes the earlier cells (at least through the one that builds `signal`,
# `kwic_idx`, `kwic_k`, `kwic_w`, `NUFFT3DOperator`, `C_op`) have been run.
# Read-only w.r.t. those objects; builds throwaway nets only.
# =============================================================================
from Modules.Regularizer import Regularizer

# ---- knobs (keep small: this is a probe, not the real run) -------------------
F_TEST  = 8      # how many of the 96 kwic frames to stack (spread across TI)
N_STEPS = 200    # self-supervised steps to watch the held-out error fall
torch.manual_seed(0)

pick = np.linspace(0, len(kwic_idx) - 1, F_TEST).astype(int)   # 8 frames in TI
print("frames (kwic idx):", list(pick),
      " TI(ms):", [int(round(ti_frames[f] * 1e3)) for f in pick])

# =============================================================================
# A) SMOKE TEST -- F as channels needs NO edit to Resnet.py:
#    Regularizer((n,n,n,F)) -> contrasts=F -> ResNet(contrasts=F) ->
#    nn.Conv3d(F -> features) ... nn.Conv3d(features -> F).  Any F works.
# =============================================================================
n_t, F_t = 32, 8
reg_t = Regularizer((n_t, n_t, n_t, F_t), regularizer="ResNet", features=16,
                    num_of_resblocks=4, kernel_size=(3, 3, 3), scale_factor=1,
                    device=device, dtype=torch.complex64)
xt = torch.randn(2 * n_t**3 * F_t, dtype=torch.float32, device=device)
yt = reg_t(xt)
ok_shape = xt.numel() == yt.numel()
yt.abs().pow(2).mean().backward()
gA = sum(float(p.grad.abs().sum()) for p in reg_t.parameters() if p.grad is not None)
print(f"[A] in={xt.numel()} out={yt.numel()} shape-match={ok_shape}  "
      f"total|grad|={gA:.3e}   (want: True and > 0)")
del reg_t, xt, yt, gA
torch.cuda.empty_cache()

# =============================================================================
# B) CURVE PROBE -- temporal self-supervision (the idea that matters)
#   input  = [b1/a1, ..., 0 at the held-out slot, ..., bF/aF]
#            (b_f/alpha_f = aliasing regrid of frame f, at true-image scale)
#   output = net(input);  take the held-out slot -> x_hat
#   loss   = || F_hold C x_hat - d_hold ||_1 / || d_hold ||_1
#            (k-space, no ground truth, O(1) by construction)
# =============================================================================
reg = Regularizer((n, n, n, F_TEST), regularizer="ResNet", features=16,
                  num_of_resblocks=4, kernel_size=(3, 3, 3), scale_factor=1,
                  device=device, dtype=torch.complex64)
print(f"[B] stack ResNet params = {sum(p.numel() for p in reg.parameters()):,}")
opt = torch.optim.Adam(reg.parameters(), lr=5e-4)

# per picked frame: windowed data d_f, raw regrid b_f, its NUFFT
# self-consistency factor alpha_f = ||F C b_f||/||d_f||, and the
# true-image-scale regrid b_f/alpha_f
Fwd, dat, regimg, alpha = {}, {}, {}, {}
with torch.no_grad():
    for f in pick:
        Ff = NUFFT3DOperator(kwic_k[f], (N_channels, n, n, n))
        dat[f] = (signal[:, kwic_idx[f]] * kwic_w[f][None, :]) \
                        .contiguous().view(torch.float32).ravel()
        bf = (C_op.H @ Ff.H @ dat[f]).view(torch.complex64).reshape(n, n, n)
        bf_flat = torch.view_as_real(bf.reshape(-1)).reshape(-1)
        alpha[f] = float(torch.linalg.norm((Ff @ C_op) @ bf_flat, 1)
                         / torch.linalg.norm(dat[f], 1))
        regimg[f] = bf / alpha[f]                 # true-image-scale regrid
        Fwd[f] = Ff
print("[B] alpha_f (regrid scale vs true image, ~ 8 x sum|coil|^2): "
      + "  ".join(f"f{f}={alpha[f]:.0f}" for f in pick))

# baseline: the "do nothing" answer = output the aliasing regrid itself.
# Its k-space self-consistency error is now O(0.1) (was ~200 in v1).
base = {}
for f in pick:
    rf_flat = torch.view_as_real(regimg[f].reshape(-1)).reshape(-1)
    base[f] = float(torch.linalg.norm((Fwd[f] @ C_op) @ rf_flat - dat[f], 1)
                    / torch.linalg.norm(dat[f], 1))
print("[B] zero-fill baseline rel k-space err (should be ~0.1, not ~200): "
      + "  ".join(f"f{f}={base[f]:.3f}" for f in pick))

losses = []
for step in range(N_STEPS):
    h = step % F_TEST                     # which slot is held out this step
    f_hold = pick[h]
    stack = torch.stack([regimg[f] for f in pick]).clone()   # (F_TEST,n,n,n)
    stack[h] = 0                          # MASK the held-out timepoint
    x = torch.view_as_real(stack.permute(1, 2, 3, 0).reshape(-1)).reshape(-1)
    opt.zero_grad()
    y = reg(x)
    y_c = torch.view_as_complex(y.reshape(-1, 2)).reshape(n, n, n, F_TEST) \
             .permute(3, 0, 1, 2)                              # (F_TEST,n,n,n)
    x_hat_flat = torch.view_as_real(y_c[h].reshape(-1)).reshape(-1)
    k_hat = (Fwd[f_hold] @ C_op) @ x_hat_flat
    l = torch.linalg.norm(k_hat - dat[f_hold], 1) / torch.linalg.norm(dat[f_hold], 1)
    l.backward()
    opt.step()
    losses.append(float(l.item()))
    if step % 25 == 0 or step == N_STEPS - 1:
        print(f"[B] step {step:3d}  held-out {f_hold:3d}  err={losses[-1]:.4f}  "
              f"cycle-mean={float(np.mean(losses[-F_TEST:])):.4f}")

print(f"\n[B] held-out rel k-space err:  start={losses[0]:.4f}  "
      f"end={losses[-1]:.4f}  mean(last16)={float(np.mean(losses[-16:])):.4f}")
print("[B] READ: loss falling toward/below the zero-fill baseline -> the")
print("    channel-mixing ResNet IS learning the shared curve.  Flat/stuck ->")
print("    channel-mixing is too weak; add the 1-D temporal conv to Resnet.py.")

# ---- see it: aliasing regrid vs the net's prediction of a held-out frame ----
mid = n // 2
f_show = pick[h]
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(np.rot90(regimg[f_show].abs()[:, :, mid].T.cpu().numpy(), 2), cmap="gray")
ax[0].set_title(f"frame {f_show} (TI={ti_frames[f_show]*1e3:.0f} ms): aliasing regrid")
ax[0].axis("off")
with torch.no_grad():
    stack = torch.stack([regimg[f] for f in pick]).clone()
    stack[h] = 0
    y = reg(torch.view_as_real(stack.permute(1, 2, 3, 0).reshape(-1)).reshape(-1))
    pred = torch.view_as_complex(y.reshape(-1, 2)).reshape(n, n, n, F_TEST) \
             .permute(3, 0, 1, 2)[h].abs()
ax[1].imshow(np.rot90(pred[:, :, mid].T.cpu().numpy(), 2), cmap="gray")
ax[1].set_title(f"net prediction from the other {F_TEST - 1} frames")
ax[1].axis("off")
ax[2].plot(losses)
ax[2].axhline(base[f_show], color="r", ls="--",
              label=f"zero-fill f{f_show}: {base[f_show]:.3f}")
ax[2].set_title("held-out rel k-space err over steps")
ax[2].set_xlabel("step"); ax[2].set_ylabel("rel l1 k-space err"); ax[2].legend()
plt.tight_layout(); plt.show()


In [ ]:
# =============================================================================
# [NEW] Z1 -- ZS-SSL DeepADMM (joint unrolled) -- SETUP   (v2: coil-correct
# k-space partitioning + shape guards; supersedes the first Z1 cell below)
# Separate additive cell; does NOT modify any earlier cell.
#
# Model = one ResNet with all F_ALL kwic frames as channels (F-as-channels;
# the probe cell above showed this needs NO change to Modules/Resnet.py).
# Training (the Z2 cell) = joint unrolled ADMM, zero-shot self-supervised
# (ZS-SSL): 3-way k-space split (train/validate/loss), K fixed partitions,
# l1 loss on the loss set, L-curve early stopping on the validate set, plus
# temporal masking (one batch frame held out, its input slot zeroed -> the
# net must predict it from the other frames = learn the shared curve).
#
# Reused read-only from earlier cells: signal, kwic_idx, kwic_k, kwic_w,
# C_op, n, N_channels, device, KbNufft, KbNufftAdjoint,
# nonuniform_fourier_transform_{forward,adjoint}, conjugate_gradient,
# recon_cg, Regularizer; and (if those cells ran) frames_admm / frames_cg /
# frames_nufft for the warm start.
#
# v2 FIX: the ZS-SSL partition is over K-SPACE POSITIONS (K per coil, shared
# across the 8 coils - as in SENSE), applied to the coil-major data as
# dat[f].view(N_channels, K, 2)[:, idx].  (v1 wrongly indexed the flat
# coil-major layout with per-coil indices -> coil 0 only + operator/data
# shape mismatch.)  Normalization is 1/(N_channels*K), the notebook's own
# convention (cell 22: nK = d.numel() // 2).
# =============================================================================
import math
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.checkpoint import checkpoint

from Modules.Regularizer import Regularizer
from Modules.Linops import LinearOperator, IdentityOperator
from Modules.NumSolvers import conjugate_gradient

# ---- knobs -------------------------------------------------------------------
F_ALL     = len(kwic_idx)   # 96 kwic frames as channels
B_FRAMES  = 4        # frames per training step (data term + loss)
U_ITERS   = 10       # ADMM unroll iterations (paper default)
CG_ITERS  = 2        # inner CG iters per ADMM step (warm-started)
N_STEPS   = 50      # max training steps (L-curve auto-stop may end earlier)
K_PART    = 10       # ZS-SSL: number of fixed k-space partitions
LR        = 5e-4
SEED      = 0
RHO       = 3e-4     # fixed (notebook finding: learnable rho collapsed to 0)
LEARN_RHO = False    # True: log-space Adam with a 1e-5 floor
WARM_MODE = "best"   # "best" = existing recon if available else regrid
                     # "cg"   = fresh CG-20 for every frame (~40 min)
                     # "regrid" = plain zero-filled regrid
USE_TL    = True     # partial transfer from deepadmm_kwic_bssfp.pt
CKPT_DIR  = "zs_ssl_ckpts"

torch.manual_seed(SEED)
np.random.seed(SEED)

# =============================================================================
# 1) Subset-k-space NUFFT operator sharing ONE global plan pair.
#    KbNufft takes k as a call-time argument, so plans are shared and only
#    the k positions change (one NUFFT3DOperator would build ~34 MB of plans
#    each; K_PART x F_ALL x 3 roles would be thousands of them).
# =============================================================================
nufft_ob = KbNufft(im_size=(n, n, n)).to(device)
adj_ob   = KbNufftAdjoint(im_size=(n, n, n)).to(device)

class KOp(LinearOperator):
    'coil-batched image (nC,n,n,n) -> k-space at k: (3, nK), cycles/voxel'
    def __init__(self, k):
        k = k.to(device=device, dtype=torch.float32)
        self.k = (-k).unsqueeze(-1)      # sign convention (matches NUFFT3DOperator)
        nC, nK = N_channels, k.shape[1]
        self.forward_shape = (nC, n, n, n, 1)
        self.adjoint_shape = (nC, nK)
        self.shape = (2 * nC * nK, 2 * nC * n**3)
        self.dtype = torch.float32
        self.device = device
    def _matvec(self, x):
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.forward_shape)
        y = nonuniform_fourier_transform_forward(self.k, x, nufft_ob=nufft_ob)
        return torch.view_as_real(y.reshape(-1)).reshape(-1)
    def _rmatvec(self, x):
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.adjoint_shape)
        y = nonuniform_fourier_transform_adjoint(self.k, x, (n, n, n), adj_ob=adj_ob)
        return torch.view_as_real(y.reshape(-1)).reshape(-1)

I_op = IdentityOperator((n, n, n))

# =============================================================================
# 2) Per-frame data (windowed, coil-major flat real) + ZS-SSL 3-way K-SPACE
#    partitions (K fixed, seeded; paper ratios: validate = 0.2 of all,
#    train = 0.4 of the rest = 0.32, loss = 0.28).  The split is over k
#    positions (per coil, shared across coils) - the standard SENSE
#    interpretation of the paper's Omega split.
# =============================================================================
dat, nKf, nSf = {}, {}, {}
for f in range(F_ALL):
    dat[f] = (signal[:, kwic_idx[f]] * kwic_w[f][None, :]) \
                    .contiguous().view(torch.float32).ravel()
    nKf[f] = kwic_k[f].shape[1]              # k positions per coil
    nSf[f] = N_channels * nKf[f]             # total complex samples
    assert nKf[f] == kwic_idx[f].numel(), (
        f"frame {f}: kwic_k has {nKf[f]} k positions but kwic_idx has "
        f"{kwic_idx[f].numel()} samples -> stale kwic arrays; re-run cell 15")
print(f"[Z1] k positions/frame (per coil): min={min(nKf.values())} "
      f"max={max(nKf.values())} (frame 0: {nKf[0]}; "
      f"complex samples/frame = {nSf[0]})")

def get_partition(f, k):
    'deterministic (train, validate, loss) k-position split of frame f, partition k'
    rng = np.random.default_rng(SEED * 1_000_003 + f * 101 + k)
    perm = rng.permutation(nKf[f])
    n_val = int(round(0.20 * nKf[f]))
    n_tr  = int(round(0.40 * (nKf[f] - n_val)))
    return (torch.as_tensor(perm[:n_tr], device=device),
            torch.as_tensor(perm[n_tr:n_tr + n_val], device=device),
            torch.as_tensor(perm[n_tr + n_val:], device=device))

def _sub(d_f, idx, Kf):
    # d_f: flat real (2 * N_channels * Kf), coil-major (coil c, k, re/im);
    # idx: selected k positions, applied to every coil.
    return d_f.view(N_channels, Kf, 2)[:, idx].reshape(-1).contiguous()

def A_ops(f, k):
    # -> (Q_tr, b_tr, A_lo, A_va, d_lo, d_va) for frame f, partition k.
    # 1/(N_channels*K)-normalized (notebook convention). Deterministic per
    # (f, k) and recomputed per call (a few ms) - no cache.
    idx_tr, idx_va, idx_lo = get_partition(f, k)
    kk = kwic_k[f].to(device)
    A_tr = KOp(kk[:, idx_tr]) @ C_op
    A_lo = KOp(kk[:, idx_lo]) @ C_op
    A_va = KOp(kk[:, idx_va]) @ C_op
    inv = 1.0 / nSf[f]
    return ((A_tr.H @ A_tr) * inv, (A_tr.H @ _sub(dat[f], idx_tr, nKf[f])) * inv,
            A_lo, A_va, _sub(dat[f], idx_lo, nKf[f]), _sub(dat[f], idx_va, nKf[f]))

# fail-fast: build + apply the (0,0) operators end to end before anything else
# (A = KOp @ C_op maps single-coil image (2 n^3) -> k-space (2 nC nK_sub);
#  Q = A^H A and b = A^H d live in single-coil image space, as in cell 22)
_Q0, _b0, _Alo0, _Ava0, _dlo0, _dva0 = A_ops(0, 0)
_ntr0 = round(0.40 * (nKf[0] - round(0.20 * nKf[0])))
_nlo0 = nKf[0] - round(0.20 * nKf[0]) - _ntr0
_x0 = torch.zeros(2 * n**3, dtype=torch.float32, device=device)
assert _b0.numel() == 2 * n**3, f"b_tr size mismatch: {_b0.numel()} != {2*n**3}"
assert (_Q0 @ _x0).numel() == 2 * n**3, "Q matvec size mismatch"
assert _dlo0.numel() == 2 * N_channels * _nlo0, f"d_lo size mismatch: {_dlo0.numel()}"
assert _dva0.numel() == 2 * N_channels * round(0.20 * nKf[0]), "d_va size mismatch"
print(f"[Z1] A_ops(0,0) ok: Q@x={(_Q0 @ _x0).numel()} (=2 n^3), b_tr={_b0.numel()}, "
      f"d_lo={_dlo0.numel()} (=2 nC x {_nlo0})")
del _Q0, _b0, _Alo0, _Ava0, _dlo0, _dva0, _x0

# =============================================================================
# 3) Model: stack ResNet (F_ALL channels) + partial transfer from the
#    per-frame denoiser (layer1/layer4 tiled across frame channels).
# =============================================================================
model = Regularizer((n, n, n, F_ALL), regularizer="ResNet", features=16,
                    num_of_resblocks=4, kernel_size=(3, 3, 3), scale_factor=1,
                    device=device, dtype=torch.complex64)
print(f"[Z1] stack ResNet (F={F_ALL} channels): "
      f"{sum(p.numel() for p in model.parameters()):,} params")
if USE_TL:
    ckpt = torch.load("deepadmm_kwic_bssfp.pt", map_location="cpu", weights_only=False)
    new_sd = {}
    for key, w in ckpt["denoiser"].items():
        if key == "layer1.convlayer.weight":      # (16,1,3,3,3)  -> (16,F_ALL,3,3,3)
            # CRITICAL: /F_ALL so the net starts as "old denoiser applied to the
            # AVERAGE of the frames" (unit gain, a contraction).  Unscaled tiling
            # sums all F frames -> ~F x gain -> the ADMM unroll diverges
            # (measured: loss x2 per ADMM iteration, 3e5 -> 1.5e12 in 5 steps).
            new_sd[key] = w.repeat(1, F_ALL, 1, 1, 1) / F_ALL
        elif key == "layer4.convlayer.weight":    # (1,16,3,3,3) -> (F_ALL,16,3,3,3)
            new_sd[key] = w.repeat(F_ALL, 1, 1, 1, 1)   # unit gain (output tiling)
        else:
            new_sd[key] = w                        # 16->16 layers: copy as-is
    model.regularizer.load_state_dict(new_sd)
    print("[Z1] partial TL from deepadmm_kwic_bssfp.pt "
          "(layer1 tiled/F_ALL -> contraction init; layer4 tiled; 16-ch copied)")

rho_log = nn.Parameter(torch.tensor(math.log(RHO), device=device)) if LEARN_RHO else None
def rho_val():
    return (rho_log.exp().clamp(min=1e-5) if rho_log is not None
            else torch.tensor(RHO, device=device))

# =============================================================================
# 4) ADMM state (per frame: flat real 2*n^3) + warm start
# =============================================================================
x_st = [None] * F_ALL
z_st = [None] * F_ALL
u_st = [torch.zeros(2 * n**3, dtype=torch.float32, device=device) for _ in range(F_ALL)]

_existing = {"admm":  globals().get("frames_admm", {}),
             "cg":    globals().get("frames_cg", {}),
             "nufft": globals().get("frames_nufft", {})}
def _to_flat(img):    # (n,n,n) complex -> flat real (2 n^3)
    return torch.view_as_real(img.reshape(-1)).reshape(-1).to(torch.float32).to(device)

t1 = time.time()
n_warm_src = {"admm": 0, "cg": 0, "nufft": 0, "regrid": 0, "cg-fresh": 0}
scale_by_src = {k: [] for k in n_warm_src}
for f in range(F_ALL):
    img, src = None, None
    if WARM_MODE in ("best", "cg"):
        for s in ("admm", "cg", "nufft"):
            if f in _existing[s]:
                img, src = _existing[s][f], s
                break
    if img is None and WARM_MODE == "cg":
        img, src = recon_cg(f, maxiter=20), "cg-fresh"
    if img is None:
        img = torch.view_as_complex((C_op.H @ KOp(kwic_k[f]).H @ dat[f])
                                    .reshape(-1, 2)).reshape(n, n, n)  # raw regrid
        src = "regrid"
    n_warm_src[src] += 1
    # --- normalize to DATA SCALE (critical): the NUFFT previews are raw
    #     adjoint NUFFTs, ~N_k x the true-image scale.  s_f = ||A x0||/||d||
    #     measures exactly that; after division x0 is k-space self-consistent
    #     (||A(x0/s_f) - d||/||d|| ~ O(aliasing), not O(10^5)).
    A_f = KOp(kwic_k[f]) @ C_op   # full-k forward in the SINGLE-image domain
                                  # (same composite as A_ops; raw KOp expects
                                  # the 8-coil image and would shape-mismatch)
    flat = _to_flat(img)
    with torch.no_grad():
        s_f = float(torch.linalg.norm(A_f @ flat, 1) / torch.linalg.norm(dat[f], 1))
    if s_f > 1e-3:
        flat = flat / s_f
    scale_by_src[src].append(s_f)
    x_st[f] = flat

# z warm start: one net forward over the full stack
model.eval()
with torch.no_grad():
    zin = torch.stack([torch.view_as_complex(x_st[f].reshape(-1, 2)).reshape(n, n, n)
                       for f in range(F_ALL)])
    y = model(torch.view_as_real(zin.permute(1, 2, 3, 0).reshape(-1)).reshape(-1))
    zall = torch.view_as_complex(y.reshape(-1, 2)).reshape(n, n, n, F_ALL).permute(3, 0, 1, 2)
    for f in range(F_ALL):
        z_st[f] = torch.view_as_real(zall[f].reshape(-1)).reshape(-1)
model.train()
print(f"[Z1] warm start: {n_warm_src}  ({time.time()-t1:.0f} s)")
for s in sorted(scale_by_src):
    if scale_by_src[s]:
        v = np.array(scale_by_src[s])
        print(f"[Z1]   '{s}' x0 scale ||A x0||/||d||: "
              f"min={v.min():.3g} median={np.median(v):.3g} max={v.max():.3g}"
              f"  (all divided out -> data scale)")
print(f"[Z1] GPU: {torch.cuda.memory_allocated()/2**30:.1f} GB allocated")

# =============================================================================
# 5) The unrolled ADMM segment (one ADMM iteration, checkpointable) + helpers
#    x_f <- CG(Q_f^train + rho I, b_f^train + rho (z_f - u_f))     (batch only)
#    z   <- net( x + u,  held-out frame's slot zeroed )            (full stack)
#    u_f <- u_f + x_f - z_f                                        (batch only)
# =============================================================================
def _admm_iter(xs, zs, us, batch, h, k, rv, batch_idx):
    xs_new = []
    for i, f in enumerate(batch):
        Q, b, *_ = A_ops(f, k)
        xs_new.append(conjugate_gradient(Q + rv * I_op, b=b + rv * (zs[i] - us[i]),
                                         x=xs[i], maxiter=CG_ITERS,
                                         disable_progressbar=True))
    zin = torch.zeros(F_ALL, n, n, n, dtype=torch.complex64, device=device)
    for f in range(F_ALL):
        if f == h:
            continue                        # TEMPORAL MASK: held-out slot is zero
        if f in batch_idx:
            v = xs_new[batch_idx[f]] + us[batch_idx[f]]
        else:
            v = x_st[f] + u_st[f]           # frozen context frames
        zin[f] = torch.view_as_complex(v.reshape(-1, 2)).reshape(n, n, n)
    y = model(torch.view_as_real(zin.permute(1, 2, 3, 0).reshape(-1)).reshape(-1))
    zall = torch.view_as_complex(y.reshape(-1, 2)).reshape(n, n, n, F_ALL).permute(3, 0, 1, 2)
    zs_new, us_new = [], []
    for i, f in enumerate(batch):
        zf = torch.view_as_real(zall[f].reshape(-1)).reshape(-1)
        zs_new.append(zf)
        us_new.append(us[i] + xs_new[i] - zf)
    return xs_new, zs_new, us_new

def unroll(batch, h, k, u_iters, use_ckpt):
    rv = rho_val()
    batch_idx = {f: i for i, f in enumerate(batch)}
    xs = [x_st[f] for f in batch]
    zs = [z_st[f] for f in batch]
    us = [u_st[f] for f in batch]
    for _ in range(u_iters):
        if use_ckpt:
            xs, zs, us = checkpoint(_admm_iter, xs, zs, us, batch, h, k, rv,
                                    batch_idx, use_reentrant=False)
        else:
            xs, zs, us = _admm_iter(xs, zs, us, batch, h, k, rv, batch_idx)
    for i, f in enumerate(batch):
        x_st[f], z_st[f], u_st[f] = xs[i], zs[i], us[i]
    return {f: xs[i] for i, f in enumerate(batch)}

def loss_l1(xrec, batch, k):
    tot = 0.0
    for f in batch:
        _, _, A_lo, _, d_lo, _ = A_ops(f, k)
        tot = tot + torch.linalg.norm((A_lo @ xrec[f]) - d_lo, 1) \
                    / torch.linalg.norm(d_lo, 1)
    return tot / len(batch)

def val_l1(xrec, batch, k):
    with torch.no_grad():
        tot = 0.0
        for f in batch:
            _, _, _, A_va, _, d_va = A_ops(f, k)
            tot = tot + torch.linalg.norm((A_va @ xrec[f]) - d_va, 1) \
                        / torch.linalg.norm(d_va, 1)
        return float(tot / len(batch))

print(f"[Z1] setup done.  rho={RHO if not LEARN_RHO else 'learned (log-space)'}  "
      f"U={U_ITERS}  CG={CG_ITERS}  B={B_FRAMES}  K={K_PART}  "
      f"partitions: train/validate/loss = 0.32/0.20/0.28 (k-positions, all coils)")


In [ ]:
# =============================================================================
# [NEW] Z2 -- ZS-SSL DeepADMM (joint unrolled) -- TRAINING
# Run AFTER the Z1 cell.
#
#  1) SMOKE run first (default below): 5 steps, U=3, no checkpoints --
#     validates the plumbing (graph, grads, timing) in ~5-10 min.
#  2) Real run: set SMOKE = False and re-run.  ~3-5 min/step, 250 steps ~
#     12-20 h.  Resumable: if the kernel dies, re-run Z2 (with SMOKE=False)
#     and it continues from the last full checkpoint.
# =============================================================================
import csv

SMOKE = False          # <- set False for the real run
N_run = 5 if SMOKE else N_STEPS
U_run = 3 if SMOKE else U_ITERS

os.makedirs(CKPT_DIR, exist_ok=True)
hist_path = os.path.join(CKPT_DIR, "lcurve.csv")
full_path = os.path.join(CKPT_DIR, "last_full.pt")

# ---- optimizer ---------------------------------------------------------------
params = list(model.parameters())
if rho_log is not None:
    params = [rho_log] + params
opt = torch.optim.Adam(params, lr=LR)

# ---- resume (real runs only) --------------------------------------------------
start_step = 0
history = []
if not SMOKE and os.path.exists(full_path):
    st = torch.load(full_path, map_location=device, weights_only=False)
    model.load_state_dict(st["model"])
    if rho_log is not None:
        rho_log.data = st["rho_log"].to(device)
    for f in range(F_ALL):
        x_st[f] = st["x_st"][f].to(device)
        z_st[f] = st["z_st"][f].to(device)
        u_st[f] = st["u_st"][f].to(device)
    start_step = int(st["step"])
    history = [tuple(r) for r in st["history"]]
    print(f"[Z2] RESUMED from step {start_step} ({len(history)} logged)")

model.train()
best_val, best_step = float("inf"), -1
stop_reason = None
t0 = time.time()

for step in range(start_step, N_run):
    k = step % K_PART
    rng_s = np.random.default_rng(SEED * 100003 + step)      # deterministic per step
    batch = [int(v) for v in rng_s.choice(F_ALL, B_FRAMES, replace=False)]
    h = batch[step % B_FRAMES]

    opt.zero_grad()
    xrec = unroll(batch, h, k, u_iters=U_run, use_ckpt=True)   # always checkpoint:
                                                              # the GPU is shared/tight
    loss = loss_l1(xrec, batch, k)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(params, 10.0)             # safety for the CG chain
    opt.step()
    val = val_l1(xrec, batch, k)

    for f in batch:                                          # state is data next step
        x_st[f] = x_st[f].detach()
        z_st[f] = z_st[f].detach()
        u_st[f] = u_st[f].detach()

    history.append((step, float(loss.item()), val, k, h))
    with torch.no_grad():   # dual canary: healthy O(0.1-5); growing fast = diverging
        u_rel = max(float(u_st[f].norm() / x_st[f].norm().clamp(min=1e-12))
                    for f in batch)
    dt = time.time() - t0
    eta = dt / (step + 1 - start_step) * (N_run - step - 1)
    print(f"[Z2] step {step:4d}  k={k}  held-out={h:3d}  loss={float(loss.item()):.4f}  "
          f"val={val:.4f}  u/x={u_rel:6.2f}   ({dt/60:.1f} min elapsed, "
          f"ETA {eta/60:.1f} min)")

    if SMOKE:
        continue

    # ---- L-curve monitoring + elbow auto-stop --------------------------------
    if len(history) >= 5:
        roll = float(np.mean([r[2] for r in history[-5:]]))
        if step >= 50:
            if roll < best_val:
                best_val, best_step = roll, step - 4
            if roll > 1.03 * best_val and (step - best_step) > 25:
                stop_reason = (f"elbow at ~step {best_step}: rolling val {roll:.4f} "
                               f"> 1.03 x best {best_val:.4f}")
                print("[Z2] " + stop_reason + "  -> stopping")
                break

    if (step + 1) % 25 == 0:
        steps_ = [r[0] for r in history]
        loss_ = [r[1] for r in history]
        val_  = [r[2] for r in history]
        def _roll(a, w=5):
            return [float(np.mean(a[max(0, i - w + 1):i + 1])) for i in range(len(a))]
        fig, ax = plt.subplots(1, 2, figsize=(12, 4))
        ax[0].plot(steps_, loss_, lw=0.7, alpha=0.5)
        ax[0].plot(steps_, _roll(loss_), "r-", lw=1.5, label="rolling-5")
        ax[0].set_title("train loss (loss-set l1, temporal-masked)"); ax[0].set_xlabel("step")
        ax[0].grid(alpha=0.3); ax[0].legend()
        ax[1].plot(steps_, val_, lw=0.7, alpha=0.5)
        ax[1].plot(steps_, _roll(val_), "g-", lw=1.5, label="rolling-5")
        if best_step > -1:
            ax[1].axvline(best_step, color="k", ls=":", label=f"best~{best_step}")
        ax[1].set_title("validate-set error (L-curve)"); ax[1].set_xlabel("step")
        ax[1].grid(alpha=0.3); ax[1].legend()
        plt.tight_layout()
        fig.savefig(os.path.join(CKPT_DIR, f"lcurve_{step+1}.png"), dpi=100)
        plt.show(); plt.close(fig)
        with open(hist_path, "w", newline="") as fh:
            wcsv = csv.writer(fh)
            wcsv.writerow(["step", "loss", "val", "k", "heldout"])
            wcsv.writerows(history)
        torch.save(model.state_dict(), os.path.join(CKPT_DIR, f"net_{step+1}.pt"))
        print(f"[Z2] L-curve saved ({step+1} steps) | GPU: "
              f"{torch.cuda.memory_allocated()/2**30:.1f} GB allocated, "
              f"{torch.cuda.memory_reserved()/2**30:.1f} GB reserved")

    if (step + 1) % 100 == 0:
        torch.save({"model": model.state_dict(),
                    "rho_log": None if rho_log is None else rho_log.data.cpu(),
                    "x_st": [t.cpu() for t in x_st],
                    "z_st": [t.cpu() for t in z_st],
                    "u_st": [t.cpu() for t in u_st],
                    "step": step + 1, "history": history,
                    "best_step": best_step, "best_val": best_val},
                   full_path)
        print(f"[Z2] full checkpoint saved ({step+1})")

print(f"\n[Z2] {'SMOKE ' if SMOKE else ''}done: {len(history)} steps in "
      f"{(time.time()-t0)/60:.1f} min" + (f"  ({stop_reason})" if stop_reason else ""))
if SMOKE:
    print("[Z2] SMOKE OK -> set SMOKE = False at the top of this cell and re-run "
          "for the real (resumable) training.")

# =============================================================================
# Final inference sweep + comparison (real run only, ~25 min)
# =============================================================================
if not SMOKE:
    print("\n[Z2] final inference sweep: full-k-space data term, no masking...")
    t1 = time.time()
    with torch.no_grad():
        B_SWEEP = 8
        for _ in range(U_ITERS):
            for f0 in range(0, F_ALL, B_SWEEP):
                bs = list(range(f0, min(f0 + B_SWEEP, F_ALL)))
                bidx = {f: i for i, f in enumerate(bs)}
                xs = [x_st[f] for f in bs]; zs = [z_st[f] for f in bs]; us = [u_st[f] for f in bs]
                xs_new = []
                for f in bs:
                    A_all = KOp(kwic_k[f]) @ C_op
                    Q = (A_all.H @ A_all) * (1.0 / nKf[f])
                    b = (A_all.H @ dat[f]) * (1.0 / nKf[f])
                    i = bidx[f]
                    xs_new.append(conjugate_gradient(Q + RHO * I_op,
                                                     b=b + RHO * (zs[i] - us[i]),
                                                     x=xs[i], maxiter=CG_ITERS,
                                                     disable_progressbar=True))
                zin = torch.zeros(F_ALL, n, n, n, dtype=torch.complex64, device=device)
                for f in range(F_ALL):
                    if f in bidx:
                        v = xs_new[bidx[f]] + us[bidx[f]]
                    else:
                        v = x_st[f] + u_st[f]
                    zin[f] = torch.view_as_complex(v.reshape(-1, 2)).reshape(n, n, n)
                y = model(torch.view_as_real(zin.permute(1, 2, 3, 0).reshape(-1)).reshape(-1))
                zall = torch.view_as_complex(y.reshape(-1, 2)).reshape(n, n, n, F_ALL).permute(3, 0, 1, 2)
                for f in bs:
                    i = bidx[f]
                    zf = torch.view_as_real(zall[f].reshape(-1)).reshape(-1)
                    x_st[f], z_st[f] = xs_new[i], zf
                    u_st[f] = us[i] + xs_new[i] - zf
    zs_frames = {f: torch.view_as_complex(x_st[f].reshape(-1, 2)).reshape(n, n, n)
                 for f in range(F_ALL)}
    torch.save(zs_frames, os.path.join(CKPT_DIR, "zs_frames.pt"))
    print(f"[Z2] sweep done ({(time.time()-t1)/60:.1f} min); "
          f"recons -> {os.path.join(CKPT_DIR, 'zs_frames.pt')}")

    # ---- comparison vs the earlier pipelines (if those cells ran) ------------
    mid = n // 2
    grid_frames = [f for f in [0, 13, 27, 40, 54, 67, 95]]
    sources = [("ZS-SSL DeepADMM (this work)", zs_frames)]
    for name, d in [("DeepADMM (per-frame)", globals().get("frames_admm")),
                    ("ContrastADMM", globals().get("frames_contrast")),
                    ("CG-20 (no net)", globals().get("frames_cg"))]:
        if d:
            sources.append((name, d))
    cols = min(len(sources), 4)
    fig, axes = plt.subplots(len(grid_frames), cols, figsize=(3.4 * cols, 3.0 * len(grid_frames)))
    for j, (name, d) in enumerate(sources[:cols]):
        for i, f in enumerate(grid_frames):
            img = d[f].abs().cpu().numpy()
            axes[i, j].imshow(np.rot90(img[:, :, mid].T, 2), cmap="gray")
            axes[i, j].set_title(f"{name}, TI={ti_frames[f]*1e3:.0f} ms" if i == 0 else
                                 f"TI={ti_frames[f]*1e3:.0f} ms", fontsize=8)
            axes[i, j].axis("off")
    plt.tight_layout(); plt.show()
    print("[Z2] ZS-SSL training complete.")
